# 12. The rolling evaluation: what the constrained portfolios realised, month by month

**Terms used in this notebook.**

| Term | Meaning |
|---|---|
| Risk-free rate | the return on a one-month US Treasury bill, the closest thing to a return with no risk |
| Excess return | a return minus the risk-free rate over the same period |
| Market capitalisation | the number of a firm's shares times their price, the firm's size in money |
| Cap-weighted | weighted by market capitalisation |
| Benchmark | in the extension, the index the portfolio tracks, the cap-weighted combination of the 49 industries built in notebook 08 |
| Variance | the average squared distance of a series from its own average; its square root is the standard deviation, the usual measure of how much a return moves; volatility is the standard deviation of returns, stated per year here by multiplying the monthly figure by the square root of 12 |
| Standard error | the uncertainty of a number estimated from a sample: the standard deviation that the estimate would show across repeated samples of the same size; for a mean it is the standard deviation of the observations divided by the square root of their number |
| Covariance matrix | the table of all variances and covariances of a set of assets, 49 by 49 here, 1,225 distinct numbers |
| Covariance estimator | a method for estimating the covariance matrix from a window of data; notebook 10 built four |
| Estimation window | the past returns an estimator sees, 120 months or three years of trading days here |
| Regime | a stretch of time over which the volatilities and co-movements of the returns stay about the same; a change of regime is a shift between two such stretches, which a window of past returns cannot see coming |
| Shrinkage | pulling an estimate part of the way towards a simpler target, which trades a little bias for less estimation error |
| RiskMetrics | the covariance estimator of J.P. Morgan's 1996 RiskMetrics document: a weighted sample covariance in which each day's weight falls by half every half-life, one year of trading days here, so that recent days count most (notebook 10's sensitivity) |
| Factor | a return series that moves many assets at once, for example the return of the whole market; the Fama-French factors are long-short portfolios built to isolate one such source each |
| Principal component | a combination of the assets, with one weight per asset, chosen so that it explains as much of the assets' total variance as a single combination can (notebook 09) |
| Minimum-variance portfolio | the fully invested portfolio with the lowest variance under a given covariance matrix; the test portfolio of the covariance literature |
| Active weight | the portfolio's weight in an asset minus the benchmark's weight in it |
| Active return | the portfolio's return minus the benchmark's return in the same month |
| Tracking error | the standard deviation of the difference between two return series, stated per year; forecast (ex-ante) when computed from a covariance matrix before the month, realised (ex-post) when measured on the active returns afterwards |
| Bias statistic | the standard deviation of the active return divided by its forecast standard deviation, over many months; 1 when the forecasts of risk are right on average, above 1 when risk is under-forecast, below 1 when over-forecast |
| Autocorrelation | the correlation of a series with its own value one period earlier |
| Variance ratio | the variance of 21-day returns divided by 21 times the variance of daily returns, measured over a window; 1 when one day's return says nothing about the next day's, above 1 under positive autocorrelation, below 1 under negative (notebook 10) |
| Long-only | said of a portfolio in which no weight is negative |
| One-way turnover | half the sum over industries of the absolute changes in weight at a rebalance; the fraction of the portfolio sold, which is also the fraction bought when the portfolio stays fully invested |
| Turnover cap | an upper limit on the one-way turnover of a rebalance; 2% a month here |
| Drifted weights | the previous month's weights after that month's returns have moved them, so that they still sum to one |
| Index turnover | the one-way turnover an index fund needs to follow its benchmark: the distance between the benchmark's new weights and its own drifted weights; zero when the weights move only because prices moved |
| Tilt | a constraint that holds named industries below their benchmark weight; here Coal, Oil and Utilities at no more than half of it |
| Exclusion | a constraint that holds named industries at zero weight; Smoke (tobacco) and Guns (weapons) here |
| Beta | how much an asset moves with a factor on average; the beta to the benchmark is how much it moves with the benchmark, one for the benchmark itself |
| Beta-neutral | a constraint that holds the portfolio's beta to the benchmark equal to one |
| Mandate | the whole set of rules the fund obeys: the tilt, the exclusions and beta neutrality, the constraint set C3 |
| Constraint set | one of the four cumulative sets C0 to C3: long-only; plus the active weight bound; plus the turnover cap; plus the mandate |
| Optimiser | the extension's routine that solves the constrained tracking-error problem (notebook 11) |
| Robust portfolio | the weights whose largest forecast tracking error across the five covariance estimates is smallest; a portfolio that trusts no single matrix (notebook 11) |
| Specification | one combination of a variant (an estimator, or the robust portfolio) and a constraint set |
| Path | the month-by-month sequence of portfolios of one specification, each built on the data before its month |
| In-sample | measured on the same data that were used to form the weights; the forecast tracking error is in-sample in this sense, because the covariance that forecasts the portfolio's risk is the one the portfolio was built on |
| Out-of-sample | measured on months the portfolio had not seen when its weights were formed; the realised tracking error is out of sample |
| Information ratio | the active return per year divided by the realised tracking error; the reward earned per unit of tracking error |
| Basis point | one hundredth of a percentage point, so 50 basis points is 0.50% |
| Vintage | the version of Ken French's files on the download date, named by the release of the CRSP database they were built from |
| Provenance | the record of what was downloaded, when, with its checksum and vintage |

**The problem, stated the way practitioners state it.** The literature judges covariance estimators by what an unconstrained minimum-variance portfolio realised after the fact; Dom, Howard, Jansen and Lohre (2024), whose framing this notebook follows, replace that portfolio with a constrained one and read its realised volatility, turnover and costs. The counterpart for an index manager is the realised tracking error of the constrained, tilted, turnover-capped portfolio against its benchmark, month after month, with each month's weights chosen before the month began, and beside it the turnover, the costs, and the question whether the risk forecast that justified each portfolio was right. Nothing in notebooks 10 and 11 can answer that; both are forecasts. This notebook runs the portfolios and measures.

**What this notebook does.** Notebooks 10 and 11 forecast. This notebook measures what happened. For every month from 1979-07 to the vintage's last month, each specification builds its portfolio on the data before the month and holds it through the month; the difference between the portfolio's return and the benchmark's is that month's active return, and the standard deviation of those active returns over 566 months is the realised tracking error, the number the forecasts of notebook 11 were about. The question the extension was built to ask is answered here, whether the choice of covariance estimator still matters, measured as realised tracking error, once the portfolio is long-only, bounded, turnover-capped and tilted. Beside it: how well the forecasts were calibrated, what the mandate's fixes of notebook 11 cost in realised terms, whether the robust portfolio did what it was built for, how often the cap had to give way, and what the tilt earned or cost in return. Two expectations (8 and 9) were added after the first run, with their expectations written before their code, and are marked as such.

**How a path is run.**

- The path starts as the index: in the first evaluation month the previous portfolio is the benchmark of that month.
- In month t the variant's covariance is estimated on the data before t (notebook 10's estimators; the robust portfolio uses all five), and the optimiser solves for the weights with the drifted previous weights as the turnover reference. Nothing from month t enters the weights.
- The portfolio earns its weights times the industries' returns of month t, the benchmark earns its weights times the same returns, and the difference is recorded with the forecast tracking error, the one-way turnover, whether the cap was relaxed and the holdings.
- 35 paths: seven variants (the four estimators on daily data, the sample covariance of 120 monthly returns, the robust portfolio, and RiskMetrics) times the four constraint sets, 28 specifications, plus seven sensitivity paths under the tilt alone (C3 without notebook 11's fixes), so that the fixes' cost is measured on realised numbers.

**Measures of a path.**

- Realised tracking error: the standard deviation of the 566 active returns, times the square root of 12.
- Forecast tracking error: the mean of the monthly forecasts, and the bias statistic (standard deviation of active return over forecast standard deviation); for the daily variants also under the ten-year variance ratio of notebook 10, which multiplies the forecast variance by the ratio measured on the benchmark's daily returns over the ten years before the month.
- Active return: the mean times 12, gross; the cost of the turnover at 50 and 100 basis points per unit of one-way turnover; the net active return; the information ratio.
- Turnover: the mean one-way turnover, the months in which the cap was relaxed and their share, and those months by calendar month; the index's own turnover beside it.
- Holdings: the mean effective number of holdings and the mean number of industries held. Realised tracking error by decade.

**What I expect to see, written before the run (`constants.py`, 4 October 2026).**

1. Count first: 566 evaluation months from 1979-07; 28 paths plus 7 sensitivity paths, each with one record per month; under C0 and C1 every path holds the benchmark in every month (active return below 1e-6 a month, a ten-thousandth of a basis point; written as 1e-10 and set to the solver's precision before the recorded run), which checks the loop, and their turnover is the index's own.
2. The extension's question: under C3 the realised tracking errors of the four daily estimators lie within 10 basis points a year of each other, and the monthly sample covariance's is above the lowest of the four (direction only; its hedges were the noisiest in notebook 11).
3. Calibration: for the four daily estimators under C3 the bias statistic of the active return lies between 0.8 and 1.25 under the 21 rule. Reported: the same under the ten-year variance ratio, and which of the two is closer to 1.
4. The robust portfolio: its realised tracking error under C3 is at most the largest of the five single estimators'. Reported: its distance from the best.
5. The fixes: for each variant the mandate's C3 realises a higher tracking error than the tilt alone (direction), by less than 15 basis points a year.
6. The cap: under C2 and C3 the share of months with a relaxed cap is below 10% for every variant; relaxations are expected in the first month of every C3 path and in July months, when French rebuilds the industry portfolios, reported by calendar month.
7. Reported, no verdict: turnover, costs, net active return and the information ratio, the effective number of holdings, realised tracking error by decade, the index's own turnover.

**Added after the first recorded run (`constants.py`, 6 October 2026), expectations fixed before their code.**

8. RiskMetrics joins as a seventh variant (the exponentially weighted covariance over five years of daily returns with a one-year half-life, notebook 10's sensitivity), because Dom, Howard, Jansen and Lohre (2024) find that time dynamics matter more than shrinkage or structure and that RiskMetrics does as well as the DCC model of Engle (2002) for minimum-variance portfolios, and because notebook 10 found it gave the lowest unconstrained volatility and the best-calibrated forecast. Expectation: under C3 its realised tracking error is at or below the sample covariance's on daily data. The robust portfolio stays the minimax over the five covariances of notebook 11, so that its path is unchanged.
9. Calibration of the forecast on its own past: in month t the forecast is multiplied by the ratio of realised to forecast tracking error over the previous 60 months of the same path, using only months before t (unchanged while fewer than 24 months are available). Expectation: under C3 the calibrated bias statistic lies between 0.85 and 1.15 for the four daily estimators. Reported: by decade, and for the robust portfolio and RiskMetrics.

Running time: about ten to fifteen minutes, most of it the 20,000 optimisations.

## Modules

`constants`, `french_loader`, `benchmark`, `rules`, `covariance` and `optimiser` are those of earlier notebooks. `evaluation` is new: the path, its records and its measures, tested on a made-up market in `tests/test_evaluation.py`.

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("cvxpy") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "cvxpy"])
import os
os.makedirs("src/bp", exist_ok=True)
open("src/bp/__init__.py", "w").close()

In [ ]:
%%writefile src/bp/constants.py
"""
Pre-committed numbers for the study.

Every band, gate, sample boundary and design parameter is defined here, one
per line, with its source beside it. A notebook may read these names; it may
not introduce numbers of its own. The replication bands (section 3) and the
design of the extension (section 5) were fixed on 10 September 2026, before any
of the rules' code was written. The expectations each notebook tests are
stated in that notebook's introduction and were written before its code; where
a level was changed after a run, the comment beside it says so, with the date
and the reason, so that a reader can judge whether the change was a
specification settled by the paper's own text or a number tuned to the results.

DGU throughout means DeMiguel, Garlappi and Uppal (2009), "Optimal Versus
Naive Diversification: How Inefficient is the 1/N Portfolio Strategy?",
Review of Financial Studies 22(5), 1915-1953. Page numbers are the journal's.
"""

# ---------------------------------------------------------------------------
# 1. Data: Ken French's library, downloaded at run time, never committed
# ---------------------------------------------------------------------------

FRENCH_BASE_URL = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"

# Every file the study reads. The vintage is whatever the library serves on the
# download date; the loader records the date, the SHA-256 of the zip and the
# CRSP vintage line printed at the top of each CSV.
FRENCH_FILES = {
    "factors3":   "F-F_Research_Data_Factors_CSV.zip",        # Mkt-RF, SMB, HML, RF, monthly from 1926-07
    "momentum":   "F-F_Momentum_Factor_CSV.zip",              # Mom, monthly from 1927-01
    "factors5":   "F-F_Research_Data_5_Factors_2x3_CSV.zip",  # Mkt-RF, SMB, HML, RMW, CMA, RF from 1963-07
    "ind10":      "10_Industry_Portfolios_CSV.zip",           # DGU "Industry" dataset
    "ind49":      "49_Industry_Portfolios_CSV.zip",           # the extension universe; carries firm counts and average size
    "size_bm25":  "25_Portfolios_5x5_CSV.zip",                # DGU "FF-1-factor" and "FF-4-factor" datasets
    # Daily files, for the extension's covariance estimators (section 5).
    "ind49_daily":    "49_Industry_Portfolios_daily_CSV.zip",        # value- and equal-weighted daily returns from 1926-07-01
    "factors3_daily": "F-F_Research_Data_Factors_daily_CSV.zip",     # Mkt-RF, SMB, HML, RF daily from 1926-07-01
    "factors5_daily": "F-F_Research_Data_5_Factors_2x3_daily_CSV.zip", # Mkt-RF, SMB, HML, RMW, CMA, RF daily from 1963-07-01
    "momentum_daily": "F-F_Momentum_Factor_daily_CSV.zip",           # Mom daily from 1926-11-03
}

FRENCH_MISSING_CODES = (-99.99, -999.0)   # French's own convention, stated at the top of every portfolio file
FRENCH_RETURNS_ARE_PERCENT = True         # the loader divides by 100 once, at load time, and nowhere else

# ---------------------------------------------------------------------------
# 2. The DGU replication
# ---------------------------------------------------------------------------

DGU_SAMPLE_START = "1963-07"   # DGU Table 2, p. 1918, for every French-sourced dataset
DGU_SAMPLE_END = "2004-11"     # DGU Table 2, p. 1918
DGU_ESTIMATION_WINDOW = 120    # months; DGU section 2, p. 1927. M = 60 is in DGU's appendix and is out of scope.
DGU_RISK_AVERSION = 1.0        # gamma in the CEQ, DGU p. 1929: "the results we report are for the case of gamma = 1"
DGU_TCOST = 0.0050             # proportional cost per unit of one-way turnover; DGU p. 1929: 50 basis points, after Balduzzi and Lynch (1999)
TCOST_SENSITIVITY = 0.0100     # the one cost sensitivity reported: double the base cost
DGU_GMINC_LOWER_BOUND_FRACTION = 0.5   # g-min-c imposes w >= a * 1 with a = 1/(2N); DGU p. 1926
DGU_BS_COV_DOF = "M - N - 2"   # the Bayes-Stein covariance divides by M - N - 2; DGU equation (5), p. 1923

# Settled 22 September 2026, a specification the paper leaves ambiguous. DGU's
# Lagrangian (8), p. 1925, shows the short-sale constraint alone; footnote 22,
# p. 1934, says the constrained rules produce "corner solutions with all wealth
# invested in a single asset", which only happens when the budget constraint
# 1'w = 1 sits inside the optimisation with gamma = 1. With the budget inside,
# mv-c on MKT/SMB/HML gives a Sharpe ratio of 0.1093 against the published
# 0.1084 and a corner solution in 257 of 377 months; without it, 0.2382 and no
# corner. The reading that reproduces the paper's tables is used; the literal
# reading is kept as rules.mv_c_cone for comparison (notebook 03).
DGU_CONSTRAINED_BUDGET_INSIDE = True

# The rules replicated: those whose only inputs are the window's sample moments
# (and, for Bayes-Stein, Jorion's prior built from them), plus the value-weighted
# market, whose Sharpe ratio is the same 0.1138 in every French-sourced column
# of Table 3 and so checks that the market series and the sample match the paper's.
DGU_RULES = ("ew", "mv", "bs", "min", "vw", "mv-c", "bs-c", "min-c", "g-min-c")
DGU_IN_SAMPLE_ROW = "mv (in sample)"   # DGU Table 3 row 2: mean-variance with M = T, the upper bound estimation error costs

# Excluded, each with the criterion. They are not built.
DGU_RULES_EXCLUDED = {
    "Bayesian diffuse-prior": "not reported by DGU (Table 1 note, p. 1917)",
    "dm": "Bayesian Data-and-Model (Pastor 2000); needs an asset-pricing prior, outside the moment-based set",
    "mp": "MacKinlay and Pastor (2000) missing-factor model; needs a maximum-likelihood factor structure, outside the moment-based set",
    "mv-min": "Kan and Zhou (2007) three-fund mixture of two rules already built; not part of the pre-specified set",
    "ew-min": "mixture of 1/N and minimum variance; not part of the pre-specified set",
    "multi-prior": "not reported by DGU (Table 1 note, p. 1917)",
}

# The datasets replicated: the four DGU built from French's site (DGU Table 2, p. 1918; Appendix A, pp. 1948-1949).
# Each is monthly excess returns over the T-bill rate from French's factor file.
DGU_DATASETS = {
    "Industry":     {"n_risky": 10, "factors": ("Mkt-RF",), "N": 11, "source": "ind10 + factors3"},
    "MKT/SMB/HML":  {"n_risky": 0,  "factors": ("Mkt-RF", "SMB", "HML"), "N": 3, "source": "factors3"},
    "FF-1-factor":  {"n_risky": 20, "factors": ("Mkt-RF",), "N": 21, "source": "size_bm25 + factors3"},
    "FF-4-factor":  {"n_risky": 20, "factors": ("Mkt-RF", "SMB", "HML", "Mom"), "N": 24, "source": "size_bm25 + factors3 + momentum"},
}
DGU_DATASETS_EXCLUDED = {
    "S&P Sectors": "Roberto Wessels' GICS sector data, not on French's site (DGU Appendix A.1)",
    "International": "MSCI country indices, not on French's site (DGU Appendix A.3)",
    "FF-3-factor": "DGU do not report it: 'very similar to FF-1-factor' (Table 3 note, p. 1931)",
}
DGU_EXCLUDE_LARGEST_SIZE_QUINTILE = True   # DGU footnote 24, p. 1949: the five largest-size portfolios of the 25 are dropped, leaving 20

# ---------------------------------------------------------------------------
# 3. Bands for the replication, written before the code ran
# ---------------------------------------------------------------------------

BAND_SHARPE_ABS = 0.03          # out-of-sample Sharpe ratio within 0.03 of DGU Table 3, per rule and dataset
CHECK_TOP3_RANKING = True        # the ranking of the top three rules within each dataset is preserved
BAND_TURNOVER_REL = 0.25        # turnover within 25% relative of DGU Table 5

# DGU Table 5 reports the unconstrained mean-variance rule's turnover at 606,594
# times that of 1/N on the Industry dataset and 10,466 times on FF-1-factor.
# Those figures are dominated by the months in which the sample covariance is
# near-singular and move by orders of magnitude when French revises a single
# return, so the band applies only where DGU's published relative turnover is
# below the level here; above it, the replicated figure is reported beside the
# published one with no pass or fail. Fixed 10 September 2026.
BAND_TURNOVER_APPLIES_BELOW_RELATIVE = 100.0

# Changed 13 September 2026, after notebook 02 had run, on its evidence: the
# out-of-sample Sharpe band applies only to rules whose published relative
# turnover (DGU Table 5, panel A) is below the same level, and the in-sample
# mean-variance row carries no verdict. Evidence: on the Industry, FF-1 and FF-4
# datasets the unconstrained mean-variance rule moved by 0.12 to 0.25 in Sharpe
# ratio under ten basis points of noise, with the published figure inside the
# range, while the 1/N rows matched to the third decimal. Before the change the
# band applied to every out-of-sample row. The level is the paper's own number,
# already used by the turnover band.
BAND_SHARPE_APPLIES_BELOW_RELATIVE = BAND_TURNOVER_APPLIES_BELOW_RELATIVE
BAND_SHARPE_DECISION_DATE = "2026-09-13"

# Sharpe ratios are compared on the same sample, 1963-07 to 2004-11, 120-month
# window, so the out-of-sample period is 1973-07 to 2004-11.
DGU_EXPECTED_T = 497           # months from 1963-07 to 2004-11 inclusive; a count, checked in notebook 01
DGU_EXPECTED_OOS = 377         # DGU_EXPECTED_T - DGU_ESTIMATION_WINDOW

# The replication extended to the latest month (notebook 07), fixed 1 October
# 2026 before the notebook's code: the nine rules on every month the vintage
# carries, reported for the paper's out-of-sample period, the months since the
# paper and the whole period. The expectations are in the notebook's introduction.
EXTENDED_NEW_PERIOD_START = "2004-12"   # the month after DGU's sample ends

# ---------------------------------------------------------------------------
# 4. DGU's analytical result and simulation (notebook 04)
# ---------------------------------------------------------------------------

# The analytical critical window (DGU Proposition 1, pp. 1937-1938; Figure 1, p. 1940).
# Panel E is the one calibrated to US stock-market data and the one the abstract quotes.
ANALYTIC_SHARPE_TANGENCY = 0.15    # S*, DGU Figure 1 panel E
ANALYTIC_SHARPE_1N = 0.12          # S_ew, DGU Figure 1 panel E
EXPECT_CRITICAL_M_LOWER = {25: 3000, 50: 6000}   # DGU p. 1941: "more than 3000 months" and "more than 6000 months"; the check is order of magnitude and direction in N

# The simulated dataset (DGU section 5.1, pp. 1941-1942).
SIM_RF_MEAN_ANNUAL = 0.02
SIM_RF_SD_ANNUAL = 0.02
SIM_FACTOR_MEAN_ANNUAL = 0.08
SIM_FACTOR_SD_ANNUAL = 0.16
SIM_ALPHA = 0.0
SIM_BETA_RANGE = (0.5, 1.5)          # loadings evenly spread
SIM_IDIO_VOL_RANGE = (0.10, 0.30)    # annual idiosyncratic volatility, uniform, cross-sectional mean 0.20
SIM_N = (10, 25, 50)
SIM_M = (120, 360, 6000)
SIM_T = 24000                        # months in one simulated history
SIM_SEED = 20260910                  # fixed so the run is reproducible; DGU's seed is unknown
# In one 24,000-month history the standard error of a monthly Sharpe ratio is
# about 1/sqrt(24000) = 0.0065, so the comparison with DGU Table 6 uses the same
# band as the empirical tables, plus the sign of (mv minus 1/N) in every cell.
BAND_SIM_SHARPE_ABS = BAND_SHARPE_ABS
# Table 6 reports 1/N and "mv (true)" as full-sample Sharpe ratios and the
# estimated rules out of sample; the comparison follows that convention. As a
# second check every rule is rerun on alternative draws of the market (seeds 1
# to SIM_SPREAD_DRAWS) and the published value is checked against their range.
SIM_SPREAD_DRAWS = 8

# Fat tails (notebook 05), an addition to the paper, fixed 30 September 2026
# before the notebook's code. The market above is rerun with every shock drawn
# from a Student-t distribution, scaled so that the mean and covariance of
# returns are exactly those of the normal market; one scale draw per month is
# shared by all shocks (a multivariate t), so an extreme month hits all assets
# at once. The normal shocks are the same draws as notebook 04's.
SIM_T_DOF = (10, 5)                  # degrees of freedom. 10: excess kurtosis 1, about a broad index's monthly returns. 5: excess kurtosis 6, about an industry portfolio's. At 4 or below the fourth moment is infinite and the standard error of a Sharpe ratio is undefined.
SIM_T_COMMON_SCALE = True            # one scale per month shared by all shocks (multivariate t); False draws one scale per shock series, which changes the shape of the month's co-movement as well as its size
# The crossing table: sample-based mean-variance against 1/N over a grid of
# windows, to locate the window at which the gap closes under each distribution.
SIM_CROSSING_WINDOWS = (120, 240, 360, 600, 1000, 2000, 3000, 4000, 6000, 9000, 12000)
FAT_TAIL_MOMENT_CHECK_ABS = 0.013    # notebook 05, expectation 1: two standard errors of a 24,000-month Sharpe ratio
FAT_TAIL_DIRECTION_SLACK = 0.01      # notebook 05, expectation 2: a cell counts as in the expected direction if Sharpe(t) <= Sharpe(normal) + this
# One test added after notebook 05's first run (6 October 2026), fixed here before
# its code and labelled in the notebook: the fat-tailed market rerun with one
# scale per shock series (common_scale False), so that an extreme month changes
# the shape of the estimated covariance as well as its level. Expectation P1a:
# the mean change in Sharpe ratio across the seven estimated rules and the nine
# cells of the Table 6 grid is more negative than under the common scale at both
# degrees of freedom. P1b: the rules whose weights depend on the shape of the
# covariance and use no means lose more on average than the other estimated rules.
FAT_TAIL_SHAPE_RULES = ("min", "min-c", "g-min-c")   # P1b: the minimum-variance rules

# Volatility that changes through time (notebook 06), the second addition to
# the paper, fixed 1 October 2026 before the notebook's code. The market of
# notebook 04 is rerun with shocks whose volatility follows a GARCH(1,1) process
# (Bollerslev 1986); the factor shock and every asset's idiosyncratic shock get
# their own process with the same parameters, so both the level and the shape
# of the covariance matrix move through time, while the unconditional mean and
# covariance are exactly those of the normal market (omega = 1 - alpha - beta).
# The expectations of notebooks 05 and 06 are in their introductions and use
# the two levels above.
SIM_GARCH_ALPHA = 0.10               # weight of last month's squared shock; a textbook monthly value
SIM_GARCH_BETA = 0.85                # weight of last month's variance; alpha + beta = 0.95 is the persistence, a half-life of a volatility shock of about 13 months
SIM_GARCH_OWN_PROCESS_PER_ASSET = True   # the factor and each idiosyncratic shock follow their own process; False would scale all shocks by one common volatility, which leaves the shape of the covariance unchanged

# ---------------------------------------------------------------------------
# 5. The extension: constrained index tracking on the 49 industries
# ---------------------------------------------------------------------------

EXT_UNIVERSE = "ind49"
EXT_ESTIMATION_WINDOW = 120        # the same window as the replication, so one convention runs through the repository
EXT_REBALANCE = "monthly"
# The extension sample starts at the first month in which all 49 industries carry
# a return (French codes the early gaps as -99.99): Hlth is the last industry to
# fill and does so in 1969-07. Notebook 01 asserts it on every run.
EXT_SAMPLE_START = "1969-07"
EXT_SAMPLE_END = None              # the latest month the vintage carries

# The benchmark (the index the portfolio tracks): the cap-weighted combination
# of the 49 industries, weights from firm count times average firm size, must
# track the market factor plus the risk-free rate within this tracking error a
# year over the full sample (notebook 08). If it does not, the derived weights
# are dropped and the benchmark is the market series.
CAPW_TE_MAX_ANNUAL = 0.0050

# Notebook 08, the cap-weight check, fixed 1 October 2026 before its code. The
# weight of industry i in month t is its firm count times its average firm size
# in French's file for month t, divided by the sum over the 49 industries;
# French measures both at the start of month t, so the weight uses nothing
# unknown when the month begins. The previous-month reading is run beside it as
# a check of this timing, and the notebook stops if it tracks more closely.
CAPW_WEIGHT_TIMING = "same_month"
CAPW_WEIGHT_TIMING_ALTERNATIVE = "previous_month"   # reported beside the chosen timing, never used as the benchmark
CAPW_MARKET_SERIES = ("Mkt-RF", "RF")                # the market series is Mkt-RF plus RF, so that both sides are plain returns
CAPW_REPORT_MONTHS = ("first", "2004-11", "last")

# The direct test of where the benchmark's gap to the market comes from
# (notebook 08, second part), fixed 1 October 2026 before its code. French forms
# the industry portfolios once a year at the end of June from the firms that
# have an SIC code then, while the market return takes in a firm from its first
# full month of trading, so between one July and the next the market holds
# firms that no industry portfolio holds. The test needs the firm list of each
# portfolio each month, which French does not publish, so it runs on the CRSP
# monthly stock file (WRDS, under Queen Mary University of London's
# subscription), the one use of data outside French's library in the study.
# The CRSP data stay on the machine that holds them: no notebook reads them,
# the repository carries none, and only the aggregates in
# DIRECT_TEST_RESULTS_FILE are published, with CRSP cited by name. The market
# universe is CRSP share codes 10 and 11 on NYSE, AMEX and NASDAQ, weighted by
# start-of-month market equity; the June-formed universe is the firms of that
# market that were in it at the end of June of the formation year with a
# positive SIC code. The script is checks/crsp_direct_test.py; its pure
# functions are in bp.direct_test.
DIRECT_TEST_SHARE_CODES = (10, 11)
DIRECT_TEST_EXCHANGE_CODES = (1, 2, 3)
DIRECT_TEST_FORMATION_MONTH = 6          # the end of June
DIRECT_TEST_MARKET_TE_MAX = 0.0020       # expectation 1: the CRSP market universe tracks French's market return within 20 basis points a year
DIRECT_TEST_MIN_CORRELATION = 0.80       # expectation 2: correlation of the predicted gap with the observed one
DIRECT_TEST_SLOPE_RANGE = (0.8, 1.2)     # expectation 2: slope of the observed gap on the predicted one
DIRECT_TEST_RESIDUAL_TE_MAX = 0.0025     # expectation 4: tracking error of what the explanation leaves, 25 basis points a year
DIRECT_TEST_RESULTS_FILE = "checks/results_crsp_direct_test.json"   # aggregates only; committed

# Constraints. Fixed 10 September 2026; the sources for the levels are in
# notebook 11, "Where the numbers come from".
ACTIVE_WEIGHT_BOUND = 0.02         # each untilted industry within 2 percentage points of its benchmark weight; an absolute cap does not fit 49 industries whose benchmark weights run from under 0.5% to over 10%
TURNOVER_CAP_MONTHLY_ONE_WAY = 0.02   # one-way turnover per rebalance, against drifted weights; 2% a month is about 24% a year, the enhanced-indexing range
TILT_INDUSTRIES = ("Coal", "Oil", "Util")   # French's column names, stripped of padding
TILT_MAX_SHARE_OF_BENCHMARK = 0.5  # each tilted industry held at no more than half its benchmark weight, a 50% reduction relative to the benchmark; a tilted industry's bounds are [0, 0.5 * benchmark], and ACTIVE_WEIGHT_BOUND does not apply to it, because the two would conflict for any industry above 4% of the benchmark
LONG_ONLY = True

# Constraint sets, cumulative, reported in this order. C3 is the mandate: the
# tilt with beta neutrality and the exclusions. Beta neutrality and the
# exclusions were added on 4 October 2026, after notebook 11's checks D1 to D4
# and before notebook 12's code, because the tilt's hedge left a beta bet
# standing when the covariance was noisy and bought tobacco and weapons. The
# tilt alone is CONSTRAINT_SET_TILT_ONLY, the C3 of the first design, reported
# beside the mandate in notebooks 11 and 12 so that the cost of the two
# additions is visible.
CONSTRAINT_SETS = {
    "C0": ("long_only",),
    "C1": ("long_only", "active_weight_bound"),
    "C2": ("long_only", "active_weight_bound", "turnover_cap"),
    "C3": ("long_only", "active_weight_bound", "turnover_cap", "tilt", "beta_neutral", "exclusion"),
}
CONSTRAINT_SET_TILT_ONLY = ("long_only", "active_weight_bound", "turnover_cap", "tilt")   # the C3 of the design of 10 September 2026
MANDATE_EXCLUSIONS = ("Smoke", "Guns")   # tobacco and weapons, the standard exclusions of Dutch institutional mandates, in French's industry names; weight zero

# The data the covariance estimators see, fixed 1 October 2026. The 120-month
# window is the paper's convention and gives 49 x 120 = 5,880 returns for the
# 1,225 entries of a 49-industry covariance matrix, fewer than five per entry.
# Practitioners estimate from daily returns (three years hold about 756 trading
# days, 30 observations per entry), impose structure, and weight recent days
# more; the extension does all three. Each estimator is built at two data
# frequencies: "monthly_120", the last EXT_ESTIMATION_WINDOW months of monthly
# excess returns, and "daily_3y", the last EXT_COV_DAILY_WINDOW_YEARS years of
# daily excess returns scaled to a monthly covariance by DAILY_TO_MONTHLY_SCALE
# (notebook 10 tests that scaling). A trading day on which any industry has no
# return (one in the sample, Softw on 1971-03-11) is left out of the window.
EXT_COV_FREQUENCIES = ("monthly_120", "daily_3y")
EXT_COV_DAILY_WINDOW_YEARS = 3
EXT_COV_DAILY_WINDOW_YEARS_SENSITIVITY = (1, 5)   # reported for the sample estimator only, no verdict
DAILY_TO_MONTHLY_SCALE = 21                        # 252 trading days a year divided by 12
EXT_COV_EWMA_HALFLIFE_DAYS = 252                   # the recency-weighted sensitivity: a half-life of one year of trading days, the RiskMetrics setting of Dom, Howard, Jansen and Lohre (2024, section 3.3.2); 126 until 3 October 2026, changed on reading that paper, before notebook 10's code

# Covariance estimators. The shrinkage target follows Dom, Howard, Jansen and
# Lohre (2024), the paper the design named as its reference, checked on
# 3 October 2026 before notebook 10's code: their linear shrinkage pulls the
# sample covariance towards a scaled identity matrix (Ledoit and Wolf 2004b),
# and their footnote 11 reports that the constant-correlation target gave
# inferior results. The same paper estimates from three years of daily returns
# and rebalances monthly, and finds that under long-only constraints the
# differences between estimators shrink, which is notebook 10's expectation 3.
COV_ESTIMATORS = ("sample", "ledoit_wolf", "factor", "pca")
LW_TARGET = "scaled_identity"          # Ledoit and Wolf (2004b), "A well-conditioned estimator for large-dimensional covariance matrices"; was "constant_correlation" until 3 October 2026
LW_TARGET_SENSITIVITY = "constant_correlation"   # Ledoit and Wolf (2004a), "Honey, I Shrunk the Sample Covariance Matrix"; reported, no verdict
FACTOR_MODEL_FACTORS = ("Mkt-RF", "SMB", "HML", "RMW", "CMA", "Mom")   # the Fama-French five factors plus momentum; the estimator is B F B' plus a diagonal of residual variances, B estimated in the same estimation window
PCA_N_COMPONENTS = 5                   # five components plus a diagonal idiosyncratic term
PCA_PC1_MARKET_CORR_MIN = 0.95         # notebook 09, expectation 2: the first component's correlation with the cap-weighted market return
PCA_USE_COVARIANCE = True              # covariance, not correlation, of monthly excess returns
PCA_SIGN_RULE = "positive market loading"
PCA_WINDOWS = {                        # three pre-specified windows; loading stability is the finding and carries no pass or fail
    "to_2019":   (None, "2019-12"),
    "2020_2022": ("2020-01", "2022-12"),
    "from_2023": ("2023-01", None),
}

# Notebook 09, the principal components of the 49 industries, fixed 1 October
# 2026 before its code; the expectations are in its introduction.
PCA_REPORT_COMPONENTS = 10
PCA_PARALLEL_DRAWS = 200
PCA_PARALLEL_PERCENTILE = 95
PCA_PARALLEL_SEED = 20260910
# Set after notebook 09 ran, as its design said: the first component's
# correlation with the benchmark fell below PCA_PC1_MARKET_CORR_MIN in the
# window from 2023 (0.837), so notebook 10's PCA estimator takes the
# benchmark's excess return as its first factor. Parallel analysis found two
# components above noise; PCA_N_COMPONENTS stays at five and the two-component
# version is reported beside it.
PCA_FIRST_COMPONENT = "market"        # notebook 10: the benchmark's excess return is the first factor; "data" would take it from the eigenvectors
PCA_N_COMPONENTS_SENSITIVITY = 2      # the parallel-analysis count on the full sample, reported beside PCA_N_COMPONENTS

# Notebook 10, the four covariance estimators, fixed 3 October 2026 before its
# code. The estimators, data frequencies, tests and expectations are stated in
# the notebook's introduction and in covariance.py.
COV_EVAL_START = "1979-07"            # 1969-07 plus 120 months; the three-year daily window starts 1972-07 and is the shorter requirement
COV_TEST_ACTIVE_DRAWS = 200           # random active-weight vectors for the calibration test
COV_TEST_SEED = 20260910
COV_LONG_ONLY_RANGE_MAX = 0.01        # expectation 3: one percentage point of annualised volatility across the four estimators on daily data
COV_SCALE_RATIO_RANGE = (0.8, 1.25)   # expectation 4: the ratio of the benchmark's realised monthly variance to 21 times its daily variance
# Three tests added after notebook 10's first run (4 October 2026), each fixed
# here before its code and labelled in the notebook: P1, a scale correction by
# the variance ratio of Lo and MacKinlay (1988), one number from the benchmark's
# daily returns that multiplies the daily covariance in place of
# DAILY_TO_MONTHLY_SCALE alone; P2, bias statistics by decade; P3, trading costs
# of the minimum-variance portfolios at EXT_COST_LEVELS.
COV_SCALE_VR_HORIZON_DAYS = DAILY_TO_MONTHLY_SCALE   # the horizon of the variance ratio: one month of trading days
COV_SCALE_VR_WINDOW_YEARS_SENSITIVITY = 10           # P1 sensitivity: the variance ratio from ten years of daily benchmark returns
COV_POST_RUN_DECADES_MIN_CLOSER = 4                  # P1a: of five decades, the corrected forecast is closer to calibrated in at least this many
COV_POST_RUN_COST_MIN_VARIANTS = 6                   # P3a and P3b: of eight main variants, the long-only portfolio beats the unconstrained one net of cost in at least this many

# Notebook 11, the optimiser, fixed 4 October 2026 before its code; the
# constraints themselves date from 10 September. The problem, the infeasible-month
# rule and the expectations are stated in the notebook's introduction and in
# optimiser.py.
OPT_SOLVER = "CLARABEL"                  # interior-point solver shipped with cvxpy; deterministic
OPT_SOLVER_OPTIONS = {"tol_gap_abs": 1e-12, "tol_gap_rel": 1e-12, "tol_feas": 1e-10}   # tighter than the solver's defaults (1e-8), which left weights 1e-4 off an independent solver's on a test problem
OPT_SOLVER_OPTIONS_ROBUST = {"tol_gap_abs": 1e-9, "tol_gap_rel": 1e-9, "tol_feas": 1e-9}   # the robust problem is written with second-order cones, on which the solver reaches 1e-9 and reports 1e-12 as inaccurate; the objective agrees with the 1e-12 run to 1e-10 on the test problems
OPT_TOL = 1e-6                           # feasibility checks on weights, bounds and turnover; also "binding" means within this of the limit
OPT_TOL_TE = 1e-6                        # expectation 2a: a forecast tracking error below this, per year, counts as zero
OPT_TILT_TE_MAX_ANNUAL = 0.01            # expectation 3: 100 basis points a year, the enhanced-indexing ex-ante limit (Robeco's composite ran at 1.06%, J.P. Morgan quotes below 2%; sources in notebook 11)
OPT_REPORT_MONTHS = ("first_eval", "2004-11", "last")   # first_eval is COV_EVAL_START, the first month with both windows
OPT_SPEED_LIMIT_STEPS = (1, 6, 12, 24)   # expectation 5: rebalances reported on the journey from 1/N to the benchmark
# Checks D1 to D4 on the practical problem behind the tilt's hedge, added after
# notebook 11's first run (4 October 2026), each fixed here before its code.
OPT_D1_MARKET_SHARE_MAX = 0.05         # D1a: the market part of the tilt's forecast active variance is below this share in every month-estimator pair
OPT_D2_TE_RISE_MAX = 0.0001            # D2a: a beta-neutral constraint raises the forecast tracking error by less than one basis point a year in every pair
OPT_D3_EXCLUDE = ("Smoke",)            # D3: the hedge's tobacco purchase refused, its cost reported
# The fixes F1 to F3 (4 October 2026, before notebook 12's code): beta neutrality
# and the exclusions join C3 (above), and a robust portfolio, the weights whose
# largest forecast active variance across the estimators' covariances is
# smallest (Goldfarb and Iyengar 2003; Tutuncu and Koenig 2004), joins the
# rolling evaluation as a variant. Its beta neutrality holds under the average
# of the five beta vectors, because one equality per covariance was infeasible
# at 1979-07 under the mandate's bounds.
OPT_FIX_RISE_MAX = 0.0015              # F2: the mandate's C3 costs less than 15 basis points a year more than the tilt alone
ROBUST_VARIANT = ("robust", "all_five")   # the robust portfolio: minimax over the five covariances
# Check D5 on the solver, added after notebook 12's first run (6 October 2026),
# fixed here before its code: at this month the index moved more than the cap
# allows, no portfolio satisfied the cap and the tilt at once, and the solver
# reported "optimal" with weights that fail the independent check. The
# optimiser therefore finds the smallest feasible turnover first and never
# poses a capped problem that has no solution; the check of every returned
# portfolio stays.
OPT_D5_MONTH = "1984-07"

# Notebook 12, the rolling evaluation, fixed 4 October 2026 before its code: one
# path per specification, each portfolio built on the data before its month,
# judged on realised tracking error, calibration of the forecasts, active
# return net of costs, turnover and holdings. The expectations are in the
# notebook's introduction.
EVAL_TE_RANGE_MAX = 0.0010           # expectation 2: 10 basis points a year across the four daily estimators
EVAL_BIAS_RANGE = (0.8, 1.25)        # expectation 3, the same band as COV_SCALE_RATIO_RANGE
EVAL_RELAX_SHARE_MAX = 0.10          # expectation 6: at most one month in ten with a relaxed cap
EVAL_START_PORTFOLIO = "benchmark"   # every path starts as the index
EVAL_BENCHMARK_TOL = 1e-6            # expectation 1: an active return below this a month counts as holding the benchmark; written as 1e-10 and set to the solver's precision (about 1e-7) before the recorded run
# Expectations 8 and 9, added after notebook 12's first run (6 October 2026),
# each fixed here before its code and labelled in the notebook. RiskMetrics
# joins as a seventh variant because Dom, Howard, Jansen and Lohre (2024) find
# that time dynamics matter more than shrinkage or factor structure; the
# calibration multiplies each forecast by the ratio of realised to forecast
# tracking error over the previous EVAL_CALIBRATION_WINDOW months of the same
# path, using only months before the forecast.
EWMA_VARIANT = ("sample", "daily_5y_ewma")   # the seventh variant: RiskMetrics, as in notebook 10's sensitivity
EVAL_CALIBRATION_WINDOW = 60                 # months of a path's own past the calibration uses; a standard deviation from 60 observations has a standard error of about 9%
EVAL_CALIBRATION_MIN = 24                    # the forecast is left as it is while fewer months are available
EVAL_CALIBRATED_BIAS_RANGE = (0.85, 1.15)    # expectation 9: the calibrated bias statistic under C3, four daily estimators

# Notebook 13, the factor attribution, fixed 6 October 2026 before its code: each
# path's monthly active return split into the contributions of the six factors
# (the active exposure to each, from the industries' betas estimated on the data
# before the month, times the factor's return in the month) plus a residual, the
# industries' own returns; the identity is the check. The paths attributed are
# the seven variants under the mandate's C3 and under the tilt alone, rebuilt
# from the modules; the expectations are in the notebook's introduction.
ATTRIBUTION_FACTORS = FACTOR_MODEL_FACTORS
ATTRIBUTION_IDENTITY_MAX_ABS_ERROR = 1e-9        # expectation 1: contributions plus residual minus active return, per month, in return units
ATTRIBUTION_SETS = ("C3", "C3 tilt only")        # the two constraint sets whose paths are attributed
ATTRIBUTION_REPRODUCTION_TOL = 0.00001           # expectation 1: the rebuilt paths reproduce notebook 12's realised tracking errors within 0.1 basis point a year
ATTRIBUTION_REFERENCE_TE = {                     # notebook 12's realised tracking errors, per year, for the reproduction check
    ("sample, daily 3y", "C3"): 0.0084654, ("ledoit_wolf, daily 3y", "C3"): 0.0084820, ("factor, daily 3y", "C3"): 0.0093449,
    ("pca, daily 3y", "C3"): 0.0091482, ("sample, monthly 120", "C3"): 0.0095761, ("robust, all five", "C3"): 0.0087358,
    ("RiskMetrics, daily 5y", "C3"): 0.0084278,
    ("sample, daily 3y", "C3 tilt only"): 0.0080276, ("ledoit_wolf, daily 3y", "C3 tilt only"): 0.0080282, ("factor, daily 3y", "C3 tilt only"): 0.0088296,
    ("pca, daily 3y", "C3 tilt only"): 0.0086722, ("sample, monthly 120", "C3 tilt only"): 0.0090924, ("robust, all five", "C3 tilt only"): 0.0082305,
    ("RiskMetrics, daily 5y", "C3 tilt only"): 0.0079698,
}
ATTRIBUTION_MARKET_EXPOSURE_MAX = 0.05           # expectation 2: the mean active exposure to Mkt-RF, in absolute value, for every path
ATTRIBUTION_SIGN_AGREEMENT_MIN = 4               # expectation 3: of the five factors other than the market, the number whose mean active exposure under the mandate has the same sign for all seven variants
ATTRIBUTION_FACTOR_SHARE_MAX = 0.5               # expectation 4: the share of the active return's variance the six factors explain, for every path
ATTRIBUTION_FACTOR_PART_MAX = 0.5                # expectation 5a: the share of the mandate's cost against the tilt alone that comes through the factor contributions, every variant
ATTRIBUTION_ARITHMETIC_TOL = 0.0002              # expectation 5b: the exclusions' factor part (the five factors other than the market) within this of the arithmetic of the inputs, per year (2 basis points), every variant

# Specification count, written down before the code runs and reported.
EXT_SPEC_COUNT = (len(COV_ESTIMATORS) + 3) * len(CONSTRAINT_SETS)   # 28 optimised specifications: four daily-based estimators, the monthly sample covariance, the robust portfolio and RiskMetrics, times four constraint sets
EXT_COST_LEVELS = (DGU_TCOST, TCOST_SENSITIVITY)              # each reported net of cost at both levels; the cost is applied after the fact and does not enter the optimisation

# ---------------------------------------------------------------------------
# 6. Output conventions
# ---------------------------------------------------------------------------

OUTPUT_DIR = "outputs"
PROVENANCE_FILE = "outputs/provenance_french.json"   # download date, URL, SHA-256 and CRSP vintage line per file
TIMEZONE = "Europe/Amsterdam"

In [ ]:
%%writefile src/bp/french_loader.py
"""
Ken French data library: download, record, parse.

The library serves each dataset as a zip holding one CSV. The CSV is not a
single table. It is a text file with between two and ten blocks, each announced by a title
line ("  Average Value Weighted Returns -- Monthly"), followed by a header row
that starts with a comma, followed by rows that start with a date: six digits
for monthly data (192607), four for annual (1927). Missing values are coded
-99.99 or -999. Returns are in percent.

This module turns that into pandas DataFrames indexed by a monthly Period, in
decimal returns, with the missing codes replaced by NaN. It also writes a
provenance record for every file it downloads: the URL, the download time, the
SHA-256 of the zip, and the CRSP vintage line French prints at the top of each
CSV. A result is only as reproducible as the vintage it was computed on, and
French revises history, so the record is part of every output.

Nothing here is committed to the repository: the zips live in a local cache
directory that .gitignore excludes.
"""
from __future__ import annotations

import hashlib
import io
import json
import re
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import requests

from . import constants as C

_DATE_ROW = re.compile(r"^\s*(\d{8}|\d{6}|\d{4})\s*,")   # daily YYYYMMDD, monthly YYYYMM, annual YYYY


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def download(key: str, cache_dir: str | Path = "data/french", force: bool = False) -> dict:
    """Fetch one French zip by its key in constants.FRENCH_FILES.

    Returns the provenance record for the file and writes the zip to cache_dir.
    A cached zip is reused unless force=True, and its record is rebuilt from the
    cached bytes, so the SHA-256 always describes the file actually on disk.
    """
    filename = C.FRENCH_FILES[key]
    url = C.FRENCH_BASE_URL + filename
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    path = cache_dir / filename
    if force or not path.exists():
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        path.write_bytes(response.content)
        fetched_now = True
    else:
        fetched_now = False
    data = path.read_bytes()
    now_utc = datetime.now(timezone.utc)
    record = {
        "key": key,
        "filename": filename,
        "url": url,
        "fetched_this_run": fetched_now,
        "download_time_utc": now_utc.isoformat(timespec="seconds"),
        "download_time_local": now_utc.astimezone(ZoneInfo(C.TIMEZONE)).isoformat(timespec="seconds"),
        "zip_sha256": sha256_bytes(data),
        "zip_bytes": len(data),
        "vintage_line": _vintage_line(data),
    }
    return record


def _csv_text(zip_bytes: bytes) -> str:
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        names = [n for n in zf.namelist() if n.lower().endswith(".csv")]
        if len(names) != 1:
            raise ValueError(f"expected one CSV in the zip, found {names}")
        return zf.read(names[0]).decode("latin-1")


def _vintage_line(zip_bytes: bytes) -> str:
    """The first line of the CSV, e.g. 'This file was created using the 202607 CRSP database.'"""
    text = _csv_text(zip_bytes)
    for line in text.splitlines():
        if line.strip():
            first = line.strip()
            return first.split(".")[0] + "." if "." in first else first
    return ""


def parse_blocks(text: str) -> list[dict]:
    """Split a French CSV into its blocks.

    Each block is returned as {"title": str, "frequency": "daily" | "monthly" |
    "annual", "frame": DataFrame} where the frame is indexed by Period (day,
    month or year) and holds floats in the file's own units (percent). The title is the last non-empty line before
    the header row that is not itself a data row.
    """
    lines = text.splitlines()
    blocks = []
    i = 0
    n = len(lines)
    while i < n:
        if _DATE_ROW.match(lines[i]):
            # Walk back to the header row (starts with a comma) and the title above it.
            header_idx = i - 1
            while header_idx >= 0 and not lines[header_idx].lstrip().startswith(","):
                header_idx -= 1
            if header_idx < 0:
                raise ValueError(f"data row at line {i} has no header row above it")
            title_idx = header_idx - 1
            while title_idx >= 0 and not lines[title_idx].strip():
                title_idx -= 1
            title = lines[title_idx].strip() if title_idx >= 0 else ""
            # The factor files and the later blocks of the size-and-value file
            # carry no title line, only prose; a sentence is not a title.
            if not title or title.endswith(".") or len(title) > 70:
                title = f"(untitled block {len(blocks)})"
            columns = [c.strip() for c in lines[header_idx].split(",")[1:]]
            rows = []
            j = i
            while j < n and _DATE_ROW.match(lines[j]):
                rows.append(lines[j])
                j += 1
            raw = pd.read_csv(io.StringIO("\n".join(rows)), header=None, index_col=0)
            raw.columns = columns[: raw.shape[1]]
            date_len = len(str(raw.index[0]).strip())
            if date_len == 8:
                idx = pd.PeriodIndex(pd.to_datetime(raw.index.astype(str), format="%Y%m%d"), freq="D")
                freq = "daily"
            elif date_len == 6:
                idx = pd.PeriodIndex([pd.Period(f"{str(d)[:4]}-{str(d)[4:6]}", freq="M") for d in raw.index])
                freq = "monthly"
            else:
                idx = pd.PeriodIndex([pd.Period(str(d), freq="Y") for d in raw.index])
                freq = "annual"
            frame = raw.astype(float)
            frame.index = idx
            frame.index.name = "period"
            blocks.append({"title": title, "frequency": freq, "frame": frame})
            i = j
        else:
            i += 1
    return blocks


def load_blocks(key: str, cache_dir: str | Path = "data/french") -> list[dict]:
    """All blocks of one file, with the file's provenance record attached to each."""
    record = download(key, cache_dir)
    text = _csv_text((Path(cache_dir) / C.FRENCH_FILES[key]).read_bytes())
    blocks = parse_blocks(text)
    for b in blocks:
        b["provenance"] = record
    return blocks


def load_monthly(key: str, block: int | str = 0, cache_dir: str | Path = "data/french",
                 as_decimal: bool = True) -> pd.DataFrame:
    """One block of one file, in decimal returns with missing codes as NaN.

    The name dates from the monthly files; the function serves the daily files
    in the same way (the block's index is then a daily Period).

    block is an integer position among the file's blocks, or a regular
    expression matched against block titles (the first match wins).
    """
    blocks = load_blocks(key, cache_dir)
    if isinstance(block, int):
        chosen = blocks[block]
    else:
        matches = [b for b in blocks if re.search(block, b["title"], flags=re.I)]
        if not matches:
            raise KeyError(f"no block in {key} matches {block!r}; titles: {[b['title'] for b in blocks]}")
        chosen = matches[0]
    frame = chosen["frame"].copy()
    for code in C.FRENCH_MISSING_CODES:
        frame = frame.mask(np.isclose(frame, code))
    if as_decimal and C.FRENCH_RETURNS_ARE_PERCENT:
        frame = frame / 100.0
    frame.attrs["title"] = chosen["title"]
    frame.attrs["provenance"] = chosen["provenance"]
    return frame


def load_counts(key: str, block: int | str, cache_dir: str | Path = "data/french") -> pd.DataFrame:
    """A block in the file's own units (firm counts, average firm size), missing codes as NaN."""
    return load_monthly(key, block, cache_dir, as_decimal=False)


def write_provenance(records: list[dict], path: str | Path = C.PROVENANCE_FILE) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    ordered = sorted(records, key=lambda r: r["key"])
    path.write_text(json.dumps(ordered, indent=2, sort_keys=True) + "\n")
    return path


def block_summary(key: str, cache_dir: str | Path = "data/french") -> pd.DataFrame:
    """One row per block: title, frequency, columns, rows, first and last period."""
    rows = []
    for k, b in enumerate(load_blocks(key, cache_dir)):
        f = b["frame"]
        rows.append({
            "file": key, "block": k, "title": b["title"], "frequency": b["frequency"],
            "columns": f.shape[1], "rows": f.shape[0],
            "first": str(f.index[0]), "last": str(f.index[-1]),
        })
    return pd.DataFrame(rows)

In [ ]:
%%writefile src/bp/benchmark.py
"""The benchmark of the extension: the cap-weighted combination of French's 49 industry portfolios.

Each industry portfolio's value-weighted return for a month is the return of the
firms in the industry, each counted in proportion to its market equity at the
start of the month. Weighting the 49 industry returns by each industry's total
market equity at the same time (firm count times average firm size, two blocks of
the same French file) gives the value-weighted return of all the firms in the 49
portfolios, which should be close to French's market return. Notebook 08 measures
how close, against the level fixed in constants.CAPW_TE_MAX_ANNUAL, and decides
whether these weights are the benchmark of notebooks 09 to 13.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from . import constants as C
from . import french_loader as fl

PERIODS_PER_YEAR = 12


def load_inputs(cache_dir: str = "data/french") -> dict[str, pd.DataFrame | pd.Series]:
    """The three monthly blocks of the 49-industry file and the market series.

    returns: value-weighted industry returns (decimal); firms: number of firms;
    size: average firm size in millions of dollars; market: Mkt-RF plus RF from
    the factors file (decimal). Missing codes are NaN in all of them.
    """
    returns = fl.load_monthly("ind49", r"Value Weighted Returns -- Monthly", cache_dir)
    firms = fl.load_counts("ind49", r"Number of Firms", cache_dir)
    size = fl.load_counts("ind49", r"Average Firm Size", cache_dir)
    f3 = fl.load_monthly("factors3", 0, cache_dir)
    excess, rf = C.CAPW_MARKET_SERIES
    market = (f3[excess] + f3[rf]).rename("market")
    return {"returns": returns, "firms": firms, "size": size, "market": market}


def market_equity(firms: pd.DataFrame, size: pd.DataFrame) -> pd.DataFrame:
    """Total market equity of each industry, in the units of the size block (millions of dollars)."""
    if not firms.index.equals(size.index) or not firms.columns.equals(size.columns):
        raise ValueError("firm counts and average sizes must share months and industries")
    return firms * size


def cap_weights(firms: pd.DataFrame, size: pd.DataFrame, timing: str = C.CAPW_WEIGHT_TIMING) -> pd.DataFrame:
    """Each industry's share of total market equity, per month; the rows sum to one.

    timing "same_month": the count and size of month t weight the returns of
    month t (French measures both at the start of the month). "previous_month":
    the count and size of month t-1 weight the returns of month t; the first
    month has no weights. Any month in which an industry's count or size is
    missing has no weights at all, so that a partial sum never passes as a
    whole-market weight.
    """
    me = market_equity(firms, size)
    if timing == "previous_month":
        me = me.shift(1)
    elif timing != "same_month":
        raise ValueError(f"unknown timing {timing!r}")
    total = me.sum(axis=1, min_count=me.shape[1])
    return me.div(total, axis=0)


def combination_return(returns: pd.DataFrame, weights: pd.DataFrame) -> pd.Series:
    """Sum over industries of weight times return, per month; NaN where any of the 49 is missing."""
    if not returns.columns.equals(weights.columns):
        raise ValueError("returns and weights must have the same industries in the same order")
    aligned = weights.reindex(returns.index)
    return (aligned * returns).sum(axis=1, min_count=returns.shape[1]).rename("combination")


def tracking_error(a: pd.Series, b: pd.Series, periods_per_year: int = PERIODS_PER_YEAR) -> float:
    """Annualised standard deviation of the per-period difference a - b, over the periods both have."""
    d = (a - b).dropna()
    if len(d) < 2:
        return float("nan")
    return float(d.std(ddof=1) * np.sqrt(periods_per_year))


def compare(a: pd.Series, b: pd.Series, periods_per_year: int = PERIODS_PER_YEAR) -> dict[str, float]:
    """Tracking error, annualised mean difference, correlation, largest absolute monthly difference and month count of a against b."""
    both = pd.concat([a, b], axis=1).dropna()
    d = both.iloc[:, 0] - both.iloc[:, 1]
    return {
        "months": int(len(d)),
        "tracking_error_annual": float(d.std(ddof=1) * np.sqrt(periods_per_year)),
        "mean_difference_annual": float(d.mean() * periods_per_year),
        "correlation": float(both.iloc[:, 0].corr(both.iloc[:, 1])) if both.std(ddof=1).gt(0).all() else float("nan"),
        "max_abs_monthly_difference": float(d.abs().max()),
        "month_of_max": str(d.abs().idxmax()),
    }


def weight_extremes(weights: pd.DataFrame, month: str | pd.Period) -> dict[str, object]:
    """The largest and smallest weights in one month, with their industries."""
    row = weights.loc[pd.Period(month, "M")].dropna()
    return {
        "month": str(pd.Period(month, "M")),
        "largest": row.idxmax(), "largest_weight": float(row.max()),
        "smallest": row.idxmin(), "smallest_weight": float(row.min()),
        "industries": int(len(row)),
    }

In [ ]:
%%writefile src/bp/rules.py
"""
Portfolio rules from DeMiguel, Garlappi and Uppal (2009), section 1.

Each rule is a function of one estimation window of monthly excess returns,
an array of shape (M, N), and returns the vector of relative weights on the N
risky assets, DGU equation (1): the mean-variance solution x = Sigma^-1 mu / gamma
normalised by the absolute value of its sum, so that the weights sum to +1 or
to -1 and the direction of the position is preserved in the few windows where
the risky assets are net short.

Notation follows the paper: mu and Sigma are the sample moments of the window,
M its length, N the number of assets. The risk aversion gamma cancels in the
normalisation for every unconstrained rule, which is why it does not appear.

Rules in this module, with the DGU abbreviation and section:
  ew       1/N, section 1.1
  mv       sample-based mean-variance, section 1.2
  bs       Bayes-Stein shrinkage, section 1.3.2, Jorion (1986)
  min      minimum variance, section 1.4.1
  vw       the value-weighted market, section 1.4.2 (all weight on the market column)
  mv-c     short-sale-constrained mean-variance, section 1.5
  bs-c     short-sale-constrained Bayes-Stein, section 1.5
  min-c    short-sale-constrained minimum variance, section 1.5
  g-min-c  minimum variance with every weight at least 1/(2N), section 1.5

The constrained rules are quadratic programmes. Each is solved exactly with
the Lawson-Hanson non-negative least squares algorithm (scipy.optimize.nnls)
after a change of variables, so there is no solver setting to tune:
  minimise  x' Sigma x / 2 - mu' x   subject to x >= 0
is, with Sigma = L L' (Cholesky), the same as minimising |L' x - L^-1 mu|^2,
a non-negative least squares problem. A budget constraint 1'w = 1 is added
as one heavily weighted extra row, then the result is renormalised; a lower
bound w >= a is a shift of variables. tests/ checks every one against an
independent solver.
"""
from __future__ import annotations

import numpy as np
from scipy.optimize import nnls

from . import constants as C


def sample_moments(R: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Sample mean and covariance (divisor M - 1) of an (M, N) window."""
    R = np.asarray(R, dtype=float)
    mu = R.mean(axis=0)
    Sigma = np.cov(R, rowvar=False, ddof=1)
    return mu, np.atleast_2d(Sigma)


def normalise(x: np.ndarray) -> np.ndarray:
    """DGU equation (1): w = x / |1'x|."""
    s = float(np.sum(x))
    if s == 0.0:
        raise ZeroDivisionError("portfolio weights sum to zero; cannot normalise")
    return x / abs(s)


def ew(R: np.ndarray) -> np.ndarray:
    n = np.asarray(R).shape[1]
    return np.full(n, 1.0 / n)


def mv(R: np.ndarray) -> np.ndarray:
    mu, Sigma = sample_moments(R)
    x = np.linalg.solve(Sigma, mu)
    return normalise(x)


def min_variance(R: np.ndarray) -> np.ndarray:
    _, Sigma = sample_moments(R)
    ones = np.ones(Sigma.shape[0])
    x = np.linalg.solve(Sigma, ones)
    return x / x.sum()


def vw(market_index: int):
    """The value-weighted market: weight one on the market column, zero elsewhere."""
    def rule(R: np.ndarray) -> np.ndarray:
        w = np.zeros(np.asarray(R).shape[1])
        w[market_index] = 1.0
        return w
    rule.__name__ = "vw"
    return rule


def in_sample_sharpe(R: np.ndarray, rule=mv) -> float:
    """DGU equation (13): the Sharpe ratio of a rule estimated once on the whole sample."""
    mu, Sigma = sample_moments(R)
    w = rule(R)
    return float(w @ mu / np.sqrt(w @ Sigma @ w))


# ---------------------------------------------------------------------------
# Bayes-Stein, DGU section 1.3.2, equations (4) and (5); Jorion (1986)
# ---------------------------------------------------------------------------

def bayes_stein_from_moments(mu: np.ndarray, Sigma_mnm2: np.ndarray, M: int) -> tuple[np.ndarray, np.ndarray, float]:
    """Jorion's predictive moments from a window's mean and its covariance with divisor M - N - 2.

    Returns (mu_bs, Sigma_bs, phi). The mean is shrunk toward the mean return
    of the minimum-variance portfolio by phi (DGU equation 5); the covariance
    is inflated for the uncertainty in the mean, Jorion's predictive variance.
    """
    N = len(mu)
    Sigma_inv_1 = np.linalg.solve(Sigma_mnm2, np.ones(N))
    w_min = Sigma_inv_1 / Sigma_inv_1.sum()
    mu_min = float(mu @ w_min)
    d = mu - mu_min
    dist = float(d @ np.linalg.solve(Sigma_mnm2, d))     # (mu - mu_min 1)' Sigma^-1 (mu - mu_min 1)
    lam = (N + 2) / dist
    phi = lam / (M + lam)                                 # equals (N+2) / ((N+2) + M * dist), DGU equation (5)
    mu_bs = (1.0 - phi) * mu + phi * mu_min
    Sigma_bs = Sigma_mnm2 * (1.0 + 1.0 / (M + lam)) + (lam / (M * (M + 1.0 + lam))) * np.outer(np.ones(N), np.ones(N)) / Sigma_inv_1.sum()
    return mu_bs, Sigma_bs, phi


def bayes_stein_moments(R: np.ndarray) -> tuple[np.ndarray, np.ndarray, float]:
    """The predictive mean and covariance of Jorion (1986), as DGU implement them, from a window of returns.

    The sample covariance here divides by M - N - 2 (DGU equation 5), the
    unbiased estimator of the inverse. Returns (mu_bs, Sigma_bs, phi).
    """
    R = np.asarray(R, dtype=float)
    M, N = R.shape
    mu = R.mean(axis=0)
    dev = R - mu
    Sigma = dev.T @ dev / (M - N - 2)
    return bayes_stein_from_moments(mu, Sigma, M)


def bs(R: np.ndarray) -> np.ndarray:
    mu_bs, Sigma_bs, _ = bayes_stein_moments(R)
    return normalise(np.linalg.solve(Sigma_bs, mu_bs))


# ---------------------------------------------------------------------------
# Constrained rules, DGU section 1.5, solved as non-negative least squares
# ---------------------------------------------------------------------------

_BUDGET_WEIGHT = 1e6   # weight of the budget row; the residual on 1'w = 1 is then below 1e-12 and the result is renormalised anyway


def _cholesky_upper(Sigma: np.ndarray) -> np.ndarray:
    """L' with Sigma = L L'. A tiny ridge keeps a near-singular window solvable."""
    N = Sigma.shape[0]
    try:
        L = np.linalg.cholesky(Sigma)
    except np.linalg.LinAlgError:
        L = np.linalg.cholesky(Sigma + 1e-12 * np.trace(Sigma) / N * np.eye(N))
    return L.T


def nonneg_mean_variance(mu: np.ndarray, Sigma: np.ndarray) -> np.ndarray:
    """argmax mu'x - x'Sigma x / 2 over x >= 0, as NNLS: minimise |L'x - L^-1 mu|^2.

    This is DGU's Lagrangian (8) read literally: the non-negativity constraint
    alone, the scale then removed by normalise(). It is kept so that the two
    readings of the equation can be compared (notebook 03); budget_mean_variance
    is the reading that reproduces the paper's tables.
    """
    Lt = _cholesky_upper(Sigma)
    b = np.linalg.solve(Lt.T, mu)           # L^-1 mu
    x, _ = nnls(Lt, b)
    return x


def budget_mean_variance(mu: np.ndarray, Sigma: np.ndarray, gamma: float = C.DGU_RISK_AVERSION) -> np.ndarray:
    """argmax mu'w - gamma w'Sigma w / 2 over w >= 0 and 1'w = 1.

    The budget constraint sits inside the optimisation, so gamma no longer
    cancels: with gamma = 1 and monthly moments the risk term is small against
    the differences in means, and the rule often puts all wealth in the asset
    with the highest estimated mean. That is the "corner solutions with all
    wealth invested in a single asset" of DGU footnote 22, and it is what
    reproduces their Tables 3 to 5 (constants.DGU_CONSTRAINED_BUDGET_INSIDE).
    As NNLS: minimise |sqrt(gamma) L'w - L^-1 mu / sqrt(gamma)|^2 with the
    budget as a heavily weighted extra row.
    """
    N = len(mu)
    Lt = _cholesky_upper(Sigma)
    A = np.vstack([np.sqrt(gamma) * Lt, _BUDGET_WEIGHT * np.ones(N)])
    b = np.concatenate([np.linalg.solve(Lt.T, mu) / np.sqrt(gamma), [_BUDGET_WEIGHT]])
    w, _ = nnls(A, b)
    return w / w.sum()


def constrained_min_variance(Sigma: np.ndarray, lower: float = 0.0) -> np.ndarray:
    """argmin w'Sigma w over w >= lower, 1'w = 1, as NNLS on v = w - lower.

    Objective |L'(v + lower 1)|^2, so the NNLS target is -L' lower 1; the budget
    1'v = 1 - N lower is appended as a heavily weighted row.
    """
    N = Sigma.shape[0]
    Lt = _cholesky_upper(Sigma)
    ones = np.ones(N)
    A = np.vstack([Lt, _BUDGET_WEIGHT * ones])
    b = np.concatenate([-Lt @ (lower * ones), [_BUDGET_WEIGHT * (1.0 - N * lower)]])
    v, _ = nnls(A, b)
    w = v + lower
    return w / w.sum()


def mv_c(R: np.ndarray) -> np.ndarray:
    mu, Sigma = sample_moments(R)
    if C.DGU_CONSTRAINED_BUDGET_INSIDE:
        return budget_mean_variance(mu, Sigma)
    return normalise(nonneg_mean_variance(mu, Sigma))


def bs_c(R: np.ndarray) -> np.ndarray:
    mu_bs, Sigma_bs, _ = bayes_stein_moments(R)
    if C.DGU_CONSTRAINED_BUDGET_INSIDE:
        return budget_mean_variance(mu_bs, Sigma_bs)
    return normalise(nonneg_mean_variance(mu_bs, Sigma_bs))


def mv_c_cone(R: np.ndarray) -> np.ndarray:
    """Equation (8) read literally, for the comparison in notebook 03; not a reported rule."""
    mu, Sigma = sample_moments(R)
    return normalise(nonneg_mean_variance(mu, Sigma))


def min_c(R: np.ndarray) -> np.ndarray:
    _, Sigma = sample_moments(R)
    return constrained_min_variance(Sigma, lower=0.0)


def g_min_c(R: np.ndarray) -> np.ndarray:
    """DGU p. 1926: minimum variance with w >= a 1, a = 1/(2N), halfway between min-c and 1/N."""
    _, Sigma = sample_moments(R)
    N = Sigma.shape[0]
    return constrained_min_variance(Sigma, lower=C.DGU_GMINC_LOWER_BOUND_FRACTION / N)


# ---------------------------------------------------------------------------
# The same rules from a window's moments, for the long simulated histories of
# notebook 04, where recomputing the covariance at every month would dominate
# the running time. Each takes (mu, Sigma, M) with Sigma the sample covariance
# with divisor M - 1, and returns the same weights as its window-based twin
# (tests/test_rules_backtest.py checks the equality).
# ---------------------------------------------------------------------------

def _to_mnm2(Sigma: np.ndarray, M: int) -> np.ndarray:
    """Rescale a divisor-(M-1) covariance to divisor M - N - 2, DGU equation (5)."""
    N = Sigma.shape[0]
    return Sigma * (M - 1) / (M - N - 2)


def ew_m(mu, Sigma, M):
    return np.full(len(mu), 1.0 / len(mu))


def mv_m(mu, Sigma, M):
    return normalise(np.linalg.solve(Sigma, mu))


def min_m(mu, Sigma, M):
    x = np.linalg.solve(Sigma, np.ones(len(mu)))
    return x / x.sum()


def bs_m(mu, Sigma, M):
    mu_bs, Sigma_bs, _ = bayes_stein_from_moments(mu, _to_mnm2(Sigma, M), M)
    return normalise(np.linalg.solve(Sigma_bs, mu_bs))


def mv_c_m(mu, Sigma, M):
    if C.DGU_CONSTRAINED_BUDGET_INSIDE:
        return budget_mean_variance(mu, Sigma)
    return normalise(nonneg_mean_variance(mu, Sigma))


def bs_c_m(mu, Sigma, M):
    mu_bs, Sigma_bs, _ = bayes_stein_from_moments(mu, _to_mnm2(Sigma, M), M)
    if C.DGU_CONSTRAINED_BUDGET_INSIDE:
        return budget_mean_variance(mu_bs, Sigma_bs)
    return normalise(nonneg_mean_variance(mu_bs, Sigma_bs))


def min_c_m(mu, Sigma, M):
    return constrained_min_variance(Sigma, lower=0.0)


def g_min_c_m(mu, Sigma, M):
    return constrained_min_variance(Sigma, lower=C.DGU_GMINC_LOWER_BOUND_FRACTION / len(mu))


MOMENT_RULES = {
    "ew": ew_m,
    "mv": mv_m,
    "bs": bs_m,
    "min": min_m,
    "mv-c": mv_c_m,
    "bs-c": bs_c_m,
    "min-c": min_c_m,
    "g-min-c": g_min_c_m,
}


# The registry. vw needs the market column and is built per dataset with vw(index).
RULES = {
    "ew": ew,
    "mv": mv,
    "bs": bs,
    "min": min_variance,
    "mv-c": mv_c,
    "bs-c": bs_c,
    "min-c": min_c,
    "g-min-c": g_min_c,
}

In [ ]:
%%writefile src/bp/covariance.py
"""Four ways to estimate a covariance matrix from a window of excess returns, and the tests that judge them (notebook 10).

Every estimator takes a window of excess returns (rows are periods, columns are
assets) and returns an N by N covariance matrix for the period that follows. The
sample covariance uses all N(N+1)/2 numbers the window offers. Ledoit-Wolf
shrinkage pulls it towards a simple target. The factor model and the
principal-component model describe it with a small number of common sources of movement
plus each asset's own variance. The tests measure how well each estimate
forecasts the variance of test portfolios in the next month (the bias
statistic) and how well the minimum-variance portfolio built from it does
(the tests of Dom, Howard, Jansen and Lohre, 2024).
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from . import constants as C
from . import rules as S

PERIODS_PER_YEAR = 12


# ---------------------------------------------------------------------------
# Estimators
# ---------------------------------------------------------------------------

def sample_cov(X: np.ndarray) -> np.ndarray:
    """The sample covariance, denominator T - 1."""
    return np.cov(np.asarray(X, dtype=float), rowvar=False, ddof=1)


def ewma_cov(X: np.ndarray, halflife: float) -> np.ndarray:
    """Exponentially weighted covariance: the weight of an observation halves every `halflife` periods, the most recent weighing most.

    The mean is weighted the same way; weights are normalised to sum to one.
    """
    X = np.asarray(X, dtype=float)
    T = X.shape[0]
    lam = 0.5 ** (1.0 / halflife)
    w = lam ** np.arange(T - 1, -1, -1)
    w = w / w.sum()
    mu = w @ X
    D = X - mu
    return (D * w[:, None]).T @ D / (1.0 - (w ** 2).sum())   # the unbiased normalisation for weighted samples


def lw_scaled_identity(X: np.ndarray) -> tuple[np.ndarray, float]:
    """Ledoit and Wolf (2004b): (1 - k) S + k m I, with S the sample covariance (denominator T), m the average
    sample variance and k the intensity that minimises the expected squared error.

    Returns the estimate and k. The intensity is b^2 / d^2 clipped to [0, 1], where d^2 is the squared
    distance between S and m I and b^2 measures how much S moves from one observation to the next.
    """
    X = np.asarray(X, dtype=float)
    T, N = X.shape
    D = X - X.mean(axis=0)
    S = D.T @ D / T
    m = np.trace(S) / N
    d2 = ((S - m * np.eye(N)) ** 2).sum() / N
    b2_bar = sum(((np.outer(x, x) - S) ** 2).sum() for x in D) / (N * T * T)
    b2 = min(b2_bar, d2)
    k = float(b2 / d2) if d2 > 0 else 0.0
    return (1.0 - k) * S + k * m * np.eye(N), k


def lw_constant_correlation(X: np.ndarray) -> tuple[np.ndarray, float]:
    """Ledoit and Wolf (2004a): shrink S towards the matrix with the same variances and one common correlation,
    the average sample correlation; the intensity follows the paper's appendix. Returns the estimate and k."""
    X = np.asarray(X, dtype=float)
    T, N = X.shape
    D = X - X.mean(axis=0)
    S = D.T @ D / T
    var = np.diag(S)
    sd = np.sqrt(var)
    corr = S / np.outer(sd, sd)
    rbar = (corr.sum() - N) / (N * (N - 1))
    F = rbar * np.outer(sd, sd)
    np.fill_diagonal(F, var)
    # pi: the sum over i, j of the estimated asymptotic variances of the entries of sqrt(T) S
    Y = D ** 2
    pi_mat = (Y.T @ Y) / T - S ** 2
    pi = pi_mat.sum()
    # rho: the diagonal part plus the covariance between the entries of S and the target's correlation structure
    term1 = ((D ** 3).T @ D) / T          # theta_{ii,ij}: average over t of x_it^3 x_jt
    help_ = term1 - var[:, None] * S      # (1/T) sum_t (x_it^2 - S_ii)(x_it x_jt - S_ij)
    theta_ii = help_
    theta_jj = help_.T
    rho_off = (rbar / 2.0) * (np.sqrt(var[None, :] / var[:, None]) * theta_ii + np.sqrt(var[:, None] / var[None, :]) * theta_jj)
    np.fill_diagonal(rho_off, 0.0)
    rho = np.trace(pi_mat) + rho_off.sum()
    gamma = ((F - S) ** 2).sum()
    k = (pi - rho) / gamma / T if gamma > 0 else 0.0
    k = float(min(1.0, max(0.0, k)))
    return (1.0 - k) * S + k * F, k


def factor_cov(X: np.ndarray, F: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """The exact factor model: X regressed on the factors F (with an intercept) over the window;
    the estimate is B Sigma_F B' plus the diagonal of residual variances. Returns the estimate and B (N by K)."""
    X = np.asarray(X, dtype=float)
    F = np.asarray(F, dtype=float)
    T, K = F.shape
    A = np.column_stack([np.ones(T), F])
    coef, *_ = np.linalg.lstsq(A, X, rcond=None)      # (K + 1) by N
    B = coef[1:].T                                     # N by K
    resid = X - A @ coef
    Sigma_F = np.cov(F, rowvar=False, ddof=1).reshape(K, K)
    resid_var = resid.var(axis=0, ddof=K + 1)
    return B @ Sigma_F @ B.T + np.diag(resid_var), B


def pca_cov(X: np.ndarray, n_components: int, market: np.ndarray | None = None) -> tuple[np.ndarray, int]:
    """The principal-component model.

    With `market` given (constants.PCA_FIRST_COMPONENT == "market"): each asset is regressed on the market
    series, and the covariance of the residuals is described by its first n_components - 1 principal
    components plus the diagonal of what they leave; the estimate is beta beta' var(market) plus that.
    Without a market series, the first n_components of X's own covariance plus the diagonal remainder.
    Returns the estimate and the number of components taken from the data.
    """
    X = np.asarray(X, dtype=float)
    if market is not None:
        m = np.asarray(market, dtype=float)
        A = np.column_stack([np.ones(len(m)), m])
        coef, *_ = np.linalg.lstsq(A, X, rcond=None)
        beta = coef[1]
        resid = X - A @ coef
        base = np.outer(beta, beta) * m.var(ddof=1)
        k = n_components - 1
        target = np.cov(resid, rowvar=False, ddof=2)
    else:
        base = 0.0
        k = n_components
        target = sample_cov(X)
    if k <= 0:
        return base + np.diag(np.diag(target)), 0
    vals, vecs = np.linalg.eigh(target)
    order = np.argsort(vals)[::-1][:k]
    V = vecs[:, order]
    L = np.clip(vals[order], 0.0, None)
    common = (V * L) @ V.T
    own = np.diag(np.clip(np.diag(target) - np.diag(common), 1e-12, None))
    return base + common + own, k


def scale_daily_to_monthly(Sigma_daily: np.ndarray, scale: float = C.DAILY_TO_MONTHLY_SCALE) -> np.ndarray:
    """A daily covariance times the number of trading days in a month; exact when daily returns do not predict the next day's."""
    return Sigma_daily * scale


def variance_ratio(x: np.ndarray, q: int = C.COV_SCALE_VR_HORIZON_DAYS) -> float:
    """The variance ratio of Lo and MacKinlay (1988): the variance of the overlapping sums of q consecutive
    values divided by q times the variance of the single values. It is 1 when the values do not predict each
    other, above 1 when a high value tends to be followed by a high one (positive autocorrelation) and below 1
    when a high value tends to be followed by a low one. Multiplying a daily covariance by q times this ratio,
    in place of q alone, gives the covariance of q-day sums that the autocorrelation implies."""
    v = np.asarray(x, dtype=float)
    if len(v) < 2 * q:
        return float("nan")
    sums = np.convolve(v, np.ones(q), mode="valid")
    return float(sums.var(ddof=1) / (q * v.var(ddof=1)))


# ---------------------------------------------------------------------------
# Windows
# ---------------------------------------------------------------------------

def monthly_window(excess_m: pd.DataFrame, month: pd.Period, months: int = C.EXT_ESTIMATION_WINDOW) -> pd.DataFrame:
    """The `months` monthly rows before `month`."""
    end = month - 1
    start = month - months
    return excess_m.loc[start:end]


def daily_window(excess_d: pd.DataFrame, month: pd.Period, years: int) -> pd.DataFrame:
    """The trading days of the 12 * years calendar months before `month`, days with any missing value left out."""
    end = (month - 1).asfreq("D", "end")
    start = (month - 12 * years).asfreq("D", "start")
    return excess_d.loc[start:end].dropna()


def daily_benchmark_excess(returns_d: pd.DataFrame, rf_d: pd.Series, weights_m: pd.DataFrame) -> pd.Series:
    """The benchmark's daily excess return: each day's industry returns weighted by that month's benchmark weights, minus the daily risk-free rate."""
    month_of_day = returns_d.index.asfreq("M")
    w = weights_m.reindex(month_of_day)
    w.index = returns_d.index
    return ((w * returns_d).sum(axis=1, min_count=returns_d.shape[1]) - rf_d.reindex(returns_d.index)).rename("benchmark_excess")


# ---------------------------------------------------------------------------
# Tests
# ---------------------------------------------------------------------------

def bias_statistic(realised: np.ndarray, predicted_var: np.ndarray) -> float:
    """The standard deviation of realised / sqrt(predicted variance): 1 when the forecasts are calibrated, above 1 when risk is under-forecast."""
    z = np.asarray(realised, dtype=float) / np.sqrt(np.asarray(predicted_var, dtype=float))
    return float(z.std(ddof=1))


def random_active_weights(n_assets: int, draws: int = C.COV_TEST_ACTIVE_DRAWS, bound: float = C.ACTIVE_WEIGHT_BOUND,
                          seed: int = C.COV_TEST_SEED) -> np.ndarray:
    """`draws` active-weight vectors: weights sum to zero and the largest absolute weight equals `bound`."""
    rng = np.random.default_rng(seed)
    U = rng.uniform(-1.0, 1.0, size=(draws, n_assets))
    U = U - U.mean(axis=1, keepdims=True)
    return bound * U / np.abs(U).max(axis=1, keepdims=True)


def min_variance_unconstrained(Sigma: np.ndarray) -> np.ndarray:
    """w proportional to Sigma^-1 1, scaled to sum to one; negative weights allowed."""
    x = np.linalg.solve(Sigma, np.ones(Sigma.shape[0]))
    return x / x.sum()


def min_variance_long_only(Sigma: np.ndarray) -> np.ndarray:
    """The long-only minimum-variance portfolio, solved exactly by non-negative least squares (rules.constrained_min_variance)."""
    return S.constrained_min_variance(Sigma, 0.0)


def condition_number(Sigma: np.ndarray) -> float:
    vals = np.linalg.eigvalsh(Sigma)
    return float(vals.max() / max(vals.min(), 1e-300))


def annualised_vol(x: pd.Series | np.ndarray) -> float:
    return float(np.std(np.asarray(x, dtype=float), ddof=1) * np.sqrt(PERIODS_PER_YEAR))


# ---------------------------------------------------------------------------
# The panels and the estimator variants, as notebooks 10 to 12 use them
# ---------------------------------------------------------------------------

class Panels:
    """Everything the estimators read: monthly and daily excess returns of the 49 industries, the six factors at both
    frequencies, the benchmark's weights and its excess return at both frequencies, and the plain monthly returns."""

    def __init__(self, start: str = C.EXT_SAMPLE_START):
        from . import french_loader as fl
        from . import benchmark as BM
        inputs = BM.load_inputs()
        f3 = fl.load_monthly("factors3", 0)
        f5 = fl.load_monthly("factors5", 0)
        mom = fl.load_monthly("momentum", 0)
        start_p = pd.Period(start, "M")
        self.returns_m = inputs["returns"].loc[start_p:]
        self.last = self.returns_m.index[-1]
        self.rf_m = f3["RF"].loc[start_p:self.last]
        self.excess_m = self.returns_m.sub(self.rf_m, axis=0)
        self.weights = BM.cap_weights(inputs["firms"].loc[start_p:], inputs["size"].loc[start_p:], C.CAPW_WEIGHT_TIMING)
        self.bench_m = (BM.combination_return(self.returns_m, self.weights) - self.rf_m).rename("benchmark_excess")
        self.factors_m = pd.concat([f5[["Mkt-RF", "SMB", "HML", "RMW", "CMA"]], mom["Mom"]], axis=1).loc[start_p:self.last]
        returns_d = fl.load_monthly("ind49_daily", r"Value Weighted Returns -- Daily").loc[start_p.asfreq("D", "start"):]
        f3d = fl.load_monthly("factors3_daily", 0)
        f5d = fl.load_monthly("factors5_daily", 0)
        momd = fl.load_monthly("momentum_daily", 0)
        self.rf_d = f3d["RF"].reindex(returns_d.index)
        self.returns_d = returns_d
        self.excess_d = returns_d.sub(self.rf_d, axis=0)
        self.factors_d = pd.concat([f5d[["Mkt-RF", "SMB", "HML", "RMW", "CMA"]], momd["Mom"]], axis=1).reindex(returns_d.index)
        self.bench_d = daily_benchmark_excess(returns_d, self.rf_d, self.weights)
        self.files = ("ind49", "factors3", "factors5", "momentum", "ind49_daily", "factors3_daily", "factors5_daily", "momentum_daily")


DAILY_YEARS = {"daily_3y": C.EXT_COV_DAILY_WINDOW_YEARS, "daily_1y": C.EXT_COV_DAILY_WINDOW_YEARS_SENSITIVITY[0],
               "daily_5y": C.EXT_COV_DAILY_WINDOW_YEARS_SENSITIVITY[1], "daily_5y_ewma": C.EXT_COV_DAILY_WINDOW_YEARS_SENSITIVITY[1]}


def estimate(name: str, freq: str, month: pd.Period, P: Panels) -> tuple[np.ndarray, float, int]:
    """One estimator variant for `month`, built on the data before it: the monthly covariance (daily ones scaled by
    DAILY_TO_MONTHLY_SCALE), the shrinkage intensity (nan for the others) and the number of observations in the window.
    The same dispatch as notebook 10's evaluation loop."""
    if freq == "monthly_120":
        X = monthly_window(P.excess_m, month); F = P.factors_m.loc[X.index]; mk = P.bench_m.loc[X.index]; scale = 1.0
    else:
        X = daily_window(P.excess_d, month, DAILY_YEARS[freq]); F = P.factors_d.loc[X.index]; mk = P.bench_d.loc[X.index]; scale = C.DAILY_TO_MONTHLY_SCALE
    Xv = X.to_numpy(); intensity = np.nan
    if name == "sample":
        S = ewma_cov(Xv, C.EXT_COV_EWMA_HALFLIFE_DAYS) if freq.endswith("ewma") else sample_cov(Xv)
    elif name == "ledoit_wolf":
        S, intensity = lw_scaled_identity(Xv)
    elif name == "ledoit_wolf_cc":
        S, intensity = lw_constant_correlation(Xv)
    elif name == "factor":
        S, _ = factor_cov(Xv, F.to_numpy())
    elif name == "pca":
        S, _ = pca_cov(Xv, C.PCA_N_COMPONENTS, market=mk.to_numpy())
    elif name == "pca_2":
        S, _ = pca_cov(Xv, C.PCA_N_COMPONENTS_SENSITIVITY, market=mk.to_numpy())
    else:
        raise KeyError(name)
    return S * scale, intensity, len(X)


# The five estimator variants of the rolling evaluation (notebook 12): the four estimators on three years of daily
# returns and the paper's convention, the sample covariance of 120 monthly returns.
EVALUATION_VARIANTS = tuple((e, "daily_3y") for e in C.COV_ESTIMATORS) + (("sample", "monthly_120"),)
# The robust portfolio (constants.ROBUST_VARIANT) is built from all five covariances at once, and RiskMetrics
# (constants.EWMA_VARIANT) is the recency-weighted sample covariance; the seven together are the variants the rolling evaluation reports.
ALL_VARIANTS = EVALUATION_VARIANTS + (C.ROBUST_VARIANT, C.EWMA_VARIANT)

In [ ]:
%%writefile src/bp/optimiser.py
"""The optimiser of the extension: the portfolio that tracks the benchmark most closely under a set of constraints (notebook 11).

For a month with benchmark weights b and a monthly covariance matrix Sigma, the
optimiser finds the weights w that minimise the forecast tracking error, the
square root of 12 (w - b)' Sigma (w - b), subject to the constraints of a named
set: the weights sum to one; no short positions (long_only); each untilted
industry within ACTIVE_WEIGHT_BOUND of its benchmark weight (active_weight_bound);
one-way turnover from the drifted previous portfolio at most
TURNOVER_CAP_MONTHLY_ONE_WAY (turnover_cap); the tilted industries at no more than
TILT_MAX_SHARE_OF_BENCHMARK of their benchmark weight (tilt); named industries
at zero (exclusion); the portfolio's beta to the benchmark equal to the benchmark's
(beta_neutral). The objective is quadratic and every constraint is linear,
so the problem is a convex quadratic program with one global minimum, solved by
cvxpy with the solver named in constants.OPT_SOLVER. Two safeguards (notebook 11,
check D5): when the turnover cap is among the constraints, the smallest feasible
one-way turnover is found first, by a linear program, so that the solver is never
handed a capped problem that has no solution (a cap below that turnover is raised
to it and the month is marked relaxed); and every solution is checked against its
constraints within constants.OPT_TOL before it is returned, whatever status the
solver reports.
"""
from __future__ import annotations

import warnings
from dataclasses import dataclass, field

import cvxpy as cp
import numpy as np

from . import constants as C

PERIODS_PER_YEAR = 12


class OptimiserError(RuntimeError):
    """The solver did not return an optimal solution, or the solution failed a check."""


@dataclass
class Solution:
    weights: np.ndarray
    active: np.ndarray                 # weights minus benchmark weights
    tracking_error: float              # forecast, per year: sqrt(12 a' Sigma a)
    turnover: float                    # one-way, against the previous portfolio (nan when there is none)
    turnover_cap_used: float | None    # the cap applied, after any relaxation
    relaxed: bool                      # True when the cap had to be raised to make the month feasible
    status: str
    binding: dict = field(default_factory=dict)   # constraint name -> count of bounds within OPT_TOL of their limit


def cholesky_factor(Sigma: np.ndarray) -> np.ndarray:
    """L with Sigma = L L', after symmetrising; a jitter of 1e-10 on the diagonal is added only if Sigma is not positive definite."""
    S = 0.5 * (Sigma + Sigma.T)
    try:
        return np.linalg.cholesky(S)
    except np.linalg.LinAlgError:
        return np.linalg.cholesky(S + 1e-10 * np.eye(S.shape[0]))


def drifted_weights(w_prev: np.ndarray, returns_prev: np.ndarray) -> np.ndarray:
    """The previous portfolio after the month's returns: w_i (1 + r_i) divided by the portfolio's own gross return."""
    grown = np.asarray(w_prev, dtype=float) * (1.0 + np.asarray(returns_prev, dtype=float))
    return grown / grown.sum()


def forecast_tracking_error(w: np.ndarray, b: np.ndarray, Sigma: np.ndarray) -> float:
    """sqrt(12 a' Sigma a) with a = w - b, in return units per year."""
    a = np.asarray(w, dtype=float) - np.asarray(b, dtype=float)
    return float(np.sqrt(max(PERIODS_PER_YEAR * a @ Sigma @ a, 0.0)))


def effective_number(w: np.ndarray) -> float:
    """1 / sum of squared weights: 49 for 1/N over 49 industries, smaller for a concentrated portfolio."""
    w = np.asarray(w, dtype=float)
    return float(1.0 / (w @ w))


def one_way_turnover(w: np.ndarray, w0: np.ndarray) -> float:
    """Half the sum of absolute weight changes: the fraction of the portfolio sold (and bought) at the rebalance."""
    return float(0.5 * np.abs(np.asarray(w, dtype=float) - np.asarray(w0, dtype=float)).sum())


def benchmark_betas(Sigma: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Each asset's beta to the benchmark under Sigma: Sigma b divided by b' Sigma b; the benchmark's own beta is one."""
    Sb = np.asarray(Sigma, dtype=float) @ np.asarray(b, dtype=float)
    return Sb / float(np.asarray(b, dtype=float) @ Sb)


def active_variance_parts(a: np.ndarray, Sigma: np.ndarray, b: np.ndarray) -> tuple[float, float]:
    """Split a' Sigma a into the market part, (a' beta)^2 times the benchmark's variance, and the rest.

    With beta = Sigma b / (b' Sigma b) the two parts add up to a' Sigma a exactly: the market part is the
    variance of the active return's projection on the benchmark, the rest is the variance of what the
    benchmark leaves unexplained."""
    a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float); Sigma = np.asarray(Sigma, dtype=float)
    beta = benchmark_betas(Sigma, b)
    var_b = float(b @ Sigma @ b)
    market = float((a @ beta) ** 2 * var_b)
    total = float(a @ Sigma @ a)
    return market, total - market


def bounds(b: np.ndarray, constraints: tuple, tilt_mask: np.ndarray, exclude_mask: np.ndarray,
           active_bound: float = C.ACTIVE_WEIGHT_BOUND, tilt_share: float = C.TILT_MAX_SHARE_OF_BENCHMARK) -> tuple[np.ndarray, np.ndarray]:
    """The lower and upper bound on each weight that the named constraints imply."""
    N = len(b)
    lower = np.full(N, -np.inf)
    upper = np.full(N, np.inf)
    if "long_only" in constraints:
        lower = np.maximum(lower, 0.0)
    if "active_weight_bound" in constraints:
        untilted = ~tilt_mask if "tilt" in constraints else np.ones(N, bool)
        lower = np.where(untilted, np.maximum(lower, b - active_bound), lower)
        upper = np.where(untilted, np.minimum(upper, b + active_bound), upper)
    if "tilt" in constraints:
        lower = np.where(tilt_mask, np.maximum(lower, 0.0), lower)
        upper = np.where(tilt_mask, np.minimum(upper, tilt_share * b), upper)
    if "exclusion" in constraints:
        lower = np.where(exclude_mask, 0.0, lower)
        upper = np.where(exclude_mask, 0.0, upper)
    return lower, upper


def _build(Ls: list, b: np.ndarray, lower: np.ndarray, upper: np.ndarray, w0: np.ndarray | None, cap: float | None,
           objective: str = "tracking_error", betas: list | None = None):
    """The cvxpy problem. `Ls` holds one Cholesky factor for the single-covariance problem and one per covariance for the robust
    one, whose objective is the largest of the forecast active variances; `betas` holds the beta vectors the
    beta-neutral constraint applies to, one per covariance."""
    N = len(b)
    w = cp.Variable(N)
    cons = [cp.sum(w) == 1]
    finite_lo = np.isfinite(lower)
    finite_up = np.isfinite(upper)
    if finite_lo.any():
        cons.append(w[finite_lo] >= lower[finite_lo])
    if finite_up.any():
        cons.append(w[finite_up] <= upper[finite_up])
    for beta in betas or []:
        cons.append(beta @ (w - b) == 0)
    turnover_expr = None
    if w0 is not None:
        # one-way turnover through explicit purchases p and sales n, w - w0 = p - n: the same quantity as half the sum of
        # absolute changes, in the form in which trades are written as buys and sells. The solver's status is not relied on
        # in either form (notebook 11, check D5: on a month with no feasible portfolio the solver reported "optimal" with
        # weights that fail the budget); check() tests every returned portfolio, and _solve poses a capped problem only
        # after the smallest feasible turnover is known.
        p = cp.Variable(N, nonneg=True)
        n = cp.Variable(N, nonneg=True)
        cons.append(w - w0 == p - n)
        turnover_expr = 0.5 * cp.sum(p + n)
        if cap is not None:
            cons.append(turnover_expr <= cap)
    if objective == "tracking_error" and len(Ls) == 1:
        obj = cp.Minimize(cp.sum_squares(Ls[0].T @ (w - b)))
    elif objective == "tracking_error":                     # robust: the largest active standard deviation across the covariances
        t = cp.Variable()                                   # written with norms (second-order cones) and in percent, which the solver handles more accurately than squares
        cons += [cp.norm(100.0 * (L.T @ (w - b)), 2) <= t for L in Ls]
        obj = cp.Minimize(t)
    else:                                                   # the smallest feasible one-way turnover
        obj = cp.Minimize(turnover_expr)
    return w, cp.Problem(obj, cons)


def solve(Sigma: np.ndarray, b: np.ndarray, constraints: tuple, w_prev: np.ndarray | None = None,
          tilt_mask: np.ndarray | None = None, exclude_mask: np.ndarray | None = None,
          turnover_cap: float = C.TURNOVER_CAP_MONTHLY_ONE_WAY, active_bound: float = C.ACTIVE_WEIGHT_BOUND,
          tilt_share: float = C.TILT_MAX_SHARE_OF_BENCHMARK, solver: str = C.OPT_SOLVER, tol: float = C.OPT_TOL) -> Solution:
    """The constrained tracking-error minimiser for one month.

    `constraints` is a tuple of names from constants.CONSTRAINT_SETS. `w_prev` is the previous portfolio after
    drift; it is required when "turnover_cap" is among the constraints. If the cap makes the month infeasible,
    the cap is raised to the smallest feasible one-way turnover and the solution says so.
    """
    return _solve([np.asarray(Sigma, dtype=float)], b, constraints, w_prev, tilt_mask, exclude_mask, turnover_cap, active_bound, tilt_share, solver, tol)


def solve_robust(Sigmas: list, b: np.ndarray, constraints: tuple, w_prev: np.ndarray | None = None,
                 tilt_mask: np.ndarray | None = None, exclude_mask: np.ndarray | None = None,
                 turnover_cap: float = C.TURNOVER_CAP_MONTHLY_ONE_WAY, active_bound: float = C.ACTIVE_WEIGHT_BOUND,
                 tilt_share: float = C.TILT_MAX_SHARE_OF_BENCHMARK, solver: str = C.OPT_SOLVER, tol: float = C.OPT_TOL) -> Solution:
    """The robust minimiser: the weights whose largest forecast active variance across the covariances in `Sigmas`
    is smallest, under the same constraints; beta neutrality, when asked, holds under the average of the covariances'
    beta vectors. The tracking error reported is the worst case across the covariances."""
    return _solve([np.asarray(S, dtype=float) for S in Sigmas], b, constraints, w_prev, tilt_mask, exclude_mask, turnover_cap, active_bound, tilt_share, solver, tol)


def _solve(Sigmas: list, b, constraints, w_prev, tilt_mask, exclude_mask, turnover_cap, active_bound, tilt_share, solver, tol) -> Solution:
    b = np.asarray(b, dtype=float)
    N = len(b)
    tilt_mask = np.zeros(N, bool) if tilt_mask is None else np.asarray(tilt_mask, bool)
    exclude_mask = np.zeros(N, bool) if exclude_mask is None else np.asarray(exclude_mask, bool)
    if "turnover_cap" in constraints and w_prev is None:
        raise OptimiserError("the turnover cap needs the previous portfolio")
    if "exclusion" in constraints and (exclude_mask is None or not np.asarray(exclude_mask, bool).any()):
        raise OptimiserError("the exclusion constraint needs a mask naming at least one industry")
    lower, upper = bounds(b, constraints, tilt_mask, exclude_mask, active_bound, tilt_share)
    if (lower > upper + tol).any():
        raise OptimiserError("a weight's lower bound exceeds its upper bound; the constraint set is contradictory for this month")
    Ls = [cholesky_factor(S) for S in Sigmas]
    w0 = None if w_prev is None else np.asarray(w_prev, dtype=float)
    cap = turnover_cap if "turnover_cap" in constraints else None
    # beta neutrality: one equality. With one covariance, its own betas; with more than one (the robust problem), the average
    # of their beta vectors, because one equality per covariance made the 1979-07 problem infeasible under the mandate's
    # bounds, and the minimax objective already penalises the market exposure that remains under each covariance.
    betas = [np.mean([benchmark_betas(S, b) for S in Sigmas], axis=0)] if "beta_neutral" in constraints else None
    relaxed = False

    options = C.OPT_SOLVER_OPTIONS if len(Ls) == 1 else C.OPT_SOLVER_OPTIONS_ROBUST
    accepted = ("optimal",) if len(Ls) == 1 else ("optimal", "optimal_inaccurate")
    if cap is not None:
        # feasibility first: the smallest one-way turnover the other constraints allow, from a linear program. A cap below
        # it would hand the solver a problem with no solution, and on such a problem the solver's status is not reliable
        # (notebook 11, check D5); so the cap is raised to that turnover, with a margin of one part in a million plus tol,
        # and the month is marked relaxed, before the capped problem is posed.
        w_t, prob_t = _build(Ls, b, lower, upper, w0, None, objective="turnover", betas=betas)
        _run(prob_t, solver, options)
        if prob_t.status not in ("optimal", "optimal_inaccurate"):
            raise OptimiserError(f"infeasible without the turnover cap: {prob_t.status}")
        check(_clean(w_t.value), b, lower, upper, w0, None, tol)      # the program's own weights pass the checks too
        if float(prob_t.value) > cap + 10 * tol:
            cap = float(prob_t.value) * (1.0 + 1e-6) + tol
            relaxed = True
    w, prob = _build(Ls, b, lower, upper, w0, cap, betas=betas)
    _run(prob, solver, options)
    x = _clean(w.value)
    if prob.status not in accepted:
        raise OptimiserError(f"solver status {prob.status} on a problem known to be feasible")
    check(x, b, lower, upper, w0, cap, tol)
    a = x - b
    binding = {}
    if "long_only" in constraints:
        binding["long_only"] = int(((x <= tol) & ~exclude_mask).sum())
    if "active_weight_bound" in constraints:
        untilted = ~tilt_mask if "tilt" in constraints else np.ones(N, bool)
        binding["active_weight_bound"] = int((untilted & (np.abs(np.abs(a) - active_bound) <= tol)).sum())
    if "tilt" in constraints:
        binding["tilt"] = int((tilt_mask & (np.abs(x - tilt_share * b) <= tol)).sum())
    to = one_way_turnover(x, w0) if w0 is not None else float("nan")
    if "turnover_cap" in constraints:
        binding["turnover_cap"] = int(abs(to - cap) <= 10 * tol)
    for beta in betas or []:
        if abs(beta @ a) > 10 * tol:
            raise OptimiserError(f"the portfolio's beta differs from the benchmark's by {beta @ a:.2e}")
    te = max(forecast_tracking_error(x, b, S) for S in Sigmas)      # the worst case across the covariances; one covariance, its own
    return Solution(weights=x, active=a, tracking_error=te, turnover=to,
                    turnover_cap_used=cap, relaxed=relaxed, status=prob.status, binding=binding)


def _run(prob, solver: str, options: dict) -> None:
    """Solve with the tight stopping rule; if the solver reports the result as inaccurate at that rule, solve again at
    its default rule (about 1e-8). A single-covariance problem is accepted only when reported optimal; the robust
    problem, written with second-order cones, is accepted as inaccurate too once the feasibility checks pass, and
    the status is recorded so that the evaluation can count such months."""
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Solution may be inaccurate")   # that case is solved again below at the default rule
        prob.solve(solver=solver, **options)
        if prob.status == "optimal_inaccurate":
            prob.solve(solver=solver)


def _clean(value) -> np.ndarray:
    if value is None:
        return np.full(0, np.nan)
    x = np.asarray(value, dtype=float).ravel()
    return np.where(np.abs(x) < 1e-12, 0.0, x)


def feasible(w: np.ndarray, b: np.ndarray, lower: np.ndarray, upper: np.ndarray, w0: np.ndarray | None, cap: float | None, tol: float) -> bool:
    """True when the weights sum to one, lie inside their bounds and respect the turnover cap, all within tol."""
    if len(w) != len(b) or not np.isfinite(w).all() or abs(w.sum() - 1.0) > tol:
        return False
    if (w < lower - tol).any() or (w > upper + tol).any():
        return False
    if cap is not None and w0 is not None and one_way_turnover(w, w0) > cap + 10 * tol:
        return False
    return True


def check(w: np.ndarray, b: np.ndarray, lower: np.ndarray, upper: np.ndarray, w0: np.ndarray | None, cap: float | None, tol: float) -> None:
    """Raise unless the weights sum to one, lie inside their bounds and respect the turnover cap, all within tol."""
    if len(w) != len(b) or not np.isfinite(w).all():
        raise OptimiserError("the solver returned no weights")
    if abs(w.sum() - 1.0) > tol:
        raise OptimiserError(f"weights sum to {w.sum():.8f}")
    if (w < lower - tol).any() or (w > upper + tol).any():
        raise OptimiserError("a weight lies outside its bounds")
    if cap is not None and w0 is not None and one_way_turnover(w, w0) > cap + 10 * tol:
        raise OptimiserError(f"turnover {one_way_turnover(w, w0):.6f} exceeds the cap {cap:.6f}")


def masks(columns, tilt=C.TILT_INDUSTRIES, exclude=()) -> tuple[np.ndarray, np.ndarray]:
    """Boolean masks over `columns` for the tilted and the excluded industries; a name that is not a column raises."""
    cols = [str(c).strip() for c in columns]
    for name in tuple(tilt) + tuple(exclude):
        if name not in cols:
            raise KeyError(f"{name} is not one of the {len(cols)} industries")
    return np.array([c in tilt for c in cols]), np.array([c in exclude for c in cols])

In [ ]:
%%writefile src/bp/evaluation.py
"""The rolling evaluation of the extension (notebook 12): one portfolio path per specification, month by month.

A path is one variant (a covariance estimator, or the robust portfolio built from all of them) under one
constraint set. It starts as the index. In every evaluation month the covariance is estimated on the data before
the month, the optimiser solves for the weights with the drifted previous weights as the turnover reference, the
portfolio and the benchmark earn the month's returns, and the active return, the forecast tracking error, the
turnover and the state of the cap are recorded. The measures of a path are computed from its records: realised
tracking error, forecast tracking error and its calibration, active return gross and net of trading costs,
turnover and relaxed-cap months, holdings.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from . import constants as C
from . import optimiser as O

PERIODS_PER_YEAR = 12


def run_path(months, Sigmas, weights: pd.DataFrame, returns: pd.DataFrame, constraints: tuple,
             tilt_mask: np.ndarray, exclude_mask: np.ndarray | None, robust: bool = False, keep_weights: bool = False) -> pd.DataFrame:
    """One path. `Sigmas` maps a month to its covariance (or to the list of covariances when `robust`).

    Returns one record per month: forecast tracking error (per year), active return, one-way turnover, the cap
    used and whether it was relaxed, the effective number of holdings, the industries held, and the weights when
    asked. The previous portfolio of the first month is the benchmark of that month (constants.EVAL_START_PORTFOLIO).
    """
    records = []
    w_prev = None
    prev_month = None
    for month in months:
        b = weights.loc[month].to_numpy()
        r = returns.loc[month].to_numpy()
        if w_prev is None:
            w0 = b.copy()
            b_drift = b.copy()
        else:
            w0 = O.drifted_weights(w_prev, returns.loc[prev_month].to_numpy())
            b_drift = O.drifted_weights(weights.loc[prev_month].to_numpy(), returns.loc[prev_month].to_numpy())
        kwargs = dict(w_prev=w0, tilt_mask=tilt_mask, exclude_mask=exclude_mask)
        sol = O.solve_robust(Sigmas[month], b, constraints, **kwargs) if robust else O.solve(Sigmas[month], b, constraints, **kwargs)
        rec = {"month": month, "forecast_te": sol.tracking_error, "active_return": float(sol.weights @ r - b @ r),
               "turnover": sol.turnover, "cap_used": sol.turnover_cap_used if "turnover_cap" in constraints else np.nan,
               "relaxed": bool(sol.relaxed), "effective_number": O.effective_number(sol.weights), "held": int((sol.weights > 1e-6).sum()),
               "status": sol.status,
               "benchmark_turnover": O.one_way_turnover(b, b_drift)}       # the index's own turnover: how far its weights moved beyond what its returns did
        if keep_weights:
            rec["weights"] = sol.weights
        records.append(rec)
        w_prev, prev_month = sol.weights, month
    return pd.DataFrame(records).set_index("month")


def realised_tracking_error(active: pd.Series | np.ndarray) -> float:
    """The standard deviation of the monthly active returns, times the square root of 12."""
    return float(np.std(np.asarray(active, dtype=float), ddof=1) * np.sqrt(PERIODS_PER_YEAR))


def bias_statistic(active: np.ndarray, forecast_te: np.ndarray, scale: np.ndarray | None = None) -> float:
    """sd(active return / forecast monthly standard deviation); `scale` multiplies the forecast variance (the variance-ratio
    correction). nan when a forecast is zero, as under the sets in which the portfolio is the benchmark."""
    f = np.asarray(forecast_te, dtype=float) / np.sqrt(PERIODS_PER_YEAR)
    if scale is not None:
        f = f * np.sqrt(np.asarray(scale, dtype=float))
    if (np.asarray(forecast_te, dtype=float) <= C.OPT_TOL_TE).any():     # a forecast of zero (the portfolio is the benchmark) has no calibration
        return float("nan")
    return float(np.std(np.asarray(active, dtype=float) / f, ddof=1))


def summarise(path: pd.DataFrame, vr_scale: pd.Series | None = None) -> dict:
    """The measures of one path, in return units per year where they are returns or risks."""
    active = path["active_return"].to_numpy()
    te = realised_tracking_error(active)
    gross = PERIODS_PER_YEAR * float(np.mean(active))
    turnover = float(path["turnover"].mean())
    out = {"realised TE": te, "forecast TE (mean)": float(path["forecast_te"].mean()),
           "bias": bias_statistic(active, path["forecast_te"].to_numpy()),
           "bias (variance ratio)": bias_statistic(active, path["forecast_te"].to_numpy(), None if vr_scale is None else vr_scale.loc[path.index].to_numpy()),
           "active return (gross)": gross, "turnover (one-way)": turnover, "benchmark turnover": float(path["benchmark_turnover"].mean()),
           "relaxed months": int(path["relaxed"].sum()), "relaxed share": float(path["relaxed"].mean()),
           "inaccurate months": int((path["status"] != "optimal").sum()),
           "effective number": float(path["effective_number"].mean()), "industries held": float(path["held"].mean())}
    for c in C.EXT_COST_LEVELS:
        bp = int(round(c * 1e4))
        cost = PERIODS_PER_YEAR * turnover * c
        out[f"cost at {bp} bp"] = cost
        out[f"net active return at {bp} bp"] = gross - cost
        out[f"information ratio at {bp} bp"] = (gross - cost) / te if te > C.OPT_TOL_TE else float("nan")   # no ratio for a portfolio that is the benchmark
    return out


def by_decade(path: pd.DataFrame, first_label: str, last_label: str) -> pd.Series:
    """Realised tracking error by decade of the evaluation month; the first and last partial decades carry the given labels."""
    def label(m):
        if m.year < 1990:
            return first_label
        if m.year >= 2020:
            return last_label
        return f"{(m.year // 10) * 10}s"
    groups = pd.Series([label(m) for m in path.index], index=path.index)
    return path.groupby(groups)["active_return"].apply(realised_tracking_error)


def calibrated_forecast(path: pd.DataFrame, window: int = C.EVAL_CALIBRATION_WINDOW, minimum: int = C.EVAL_CALIBRATION_MIN) -> pd.Series:
    """The forecast tracking error of month t multiplied by the ratio of realised to forecast tracking error over the
    previous `window` months of the same path (square root of the mean squared active return over the mean forecast
    variance), using only months before t; unchanged while fewer than `minimum` months are available."""
    f_var = (path["forecast_te"] / np.sqrt(PERIODS_PER_YEAR)) ** 2
    a2 = path["active_return"] ** 2
    num = a2.shift(1).rolling(window, min_periods=minimum).mean()
    den = f_var.shift(1).rolling(window, min_periods=minimum).mean()
    ratio = np.sqrt(num / den)
    ratio = ratio.where(den > 0).fillna(1.0)
    return path["forecast_te"] * ratio

## Count first: the months, the paths, the variance ratios

In [ ]:
import sys
sys.path.insert(0, "src")
import time
import numpy as np
import pandas as pd
from bp import constants as C
from bp import french_loader as fl
from bp import covariance as CV
from bp import optimiser as O
from bp import evaluation as E

pd.set_option("display.width", 230)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 40)

P = CV.Panels()
weights, returns_m = P.weights, P.returns_m
months = pd.period_range(C.COV_EVAL_START, P.last, freq="M")
industries = [c.strip() for c in weights.columns]
tilt_mask, _ = O.masks(weights.columns)
_, mandate_mask = O.masks(weights.columns, exclude=C.MANDATE_EXCLUSIONS)
SETS = dict(C.CONSTRAINT_SETS)
SENSITIVITY_SET = ("C3 tilt only", C.CONSTRAINT_SET_TILT_ONLY)
VARIANTS = list(CV.ALL_VARIANTS)
LABEL = {v: ("robust, all five" if v == C.ROBUST_VARIANT else ("RiskMetrics, daily 5y" if v == C.EWMA_VARIANT else f"{v[0]}, {v[1].replace('_', ' ')}")) for v in VARIANTS}

# Expectation 1, the counts
assert len(months) == 566, len(months)
n_paths = len(VARIANTS) * len(SETS)
assert n_paths == C.EXT_SPEC_COUNT == 28
# the ten-year variance ratio of the benchmark's daily excess return before each month (notebook 10's correction)
vr10 = pd.Series({m: CV.variance_ratio(P.bench_d.loc[CV.daily_window(P.excess_d, m, C.COV_SCALE_VR_WINDOW_YEARS_SENSITIVITY).index].to_numpy(), C.COV_SCALE_VR_HORIZON_DAYS) for m in months})
provenance = [fl.download(key) for key in P.files]
fl.write_provenance(provenance)
vintage = provenance[0]["vintage_line"].split("the ")[1].split(" ")[0]
print(f"{len(months)} evaluation months, {months[0]} to {months[-1]}; {n_paths} specifications ({len(VARIANTS)} variants, five estimators, the robust portfolio and RiskMetrics, times {len(SETS)} constraint sets) plus {len(VARIANTS)} sensitivity paths under the tilt alone")
print(f"ten-year variance ratio of the benchmark before each month: {vr10.min():.2f} to {vr10.max():.2f}, mean {vr10.mean():.2f}")
print(f"CRSP vintage of the eight files: {vintage}")

## The covariances, once per month

Each of the five estimator variants and RiskMetrics is built once per month and shared by every constraint set; the robust portfolio uses the five estimator covariances, as in notebook 11.

In [ ]:
t0 = time.time()
Sigmas = {v: {} for v in CV.EVALUATION_VARIANTS + (C.EWMA_VARIANT,)}
for i, month in enumerate(months):
    for v in CV.EVALUATION_VARIANTS + (C.EWMA_VARIANT,):
        Sigmas[v][month], _, _ = CV.estimate(*v, month, P)
    if i % 100 == 0:
        print(f"{month}: {time.time() - t0:.0f}s")
Sigmas[C.ROBUST_VARIANT] = {m: [Sigmas[v][m] for v in CV.EVALUATION_VARIANTS] for m in months}   # the robust portfolio's set: the five estimator covariances of notebook 11
print(f"{len(months) * (len(CV.EVALUATION_VARIANTS) + 1)} covariance matrices in {time.time() - t0:.0f}s")

## The paths

One path per specification, and the seven sensitivity paths. Each month's optimisation uses the drifted previous weights of its own path.

In [ ]:
paths = {}
t0 = time.time()
for v in VARIANTS:
    for cs, names in list(SETS.items()) + [SENSITIVITY_SET]:
        t1 = time.time()
        paths[(v, cs)] = E.run_path(months, Sigmas[v], weights, returns_m, names, tilt_mask,
                                    mandate_mask if "exclusion" in names else None, robust=(v == C.ROBUST_VARIANT), keep_weights=True)
        print(f"{LABEL[v]:>24}  {cs:<13} {time.time() - t1:5.0f}s")
print(f"{len(paths)} paths in {time.time() - t0:.0f}s")

# Expectation 1, the loop check: under C0 and C1 every path is the benchmark
hold = [(v, cs) for v in VARIANTS for cs in ("C0", "C1") if np.abs(paths[(v, cs)]["active_return"]).max() < C.EVAL_BENCHMARK_TOL]
largest_dev = max(np.abs(paths[(v, cs)]["active_return"]).max() for v in VARIANTS for cs in ("C0", "C1"))
index_turnover = paths[(VARIANTS[0], "C0")]["benchmark_turnover"]
print()
print(f"expectation 1: {len(months)} months; {len(paths)} paths with {set(len(p) for p in paths.values())} records each; "
      f"the benchmark is held in every month under C0 and C1 in {len(hold)} of {2 * len(VARIANTS)} paths (largest active return {largest_dev:.1e} a month, level {C.EVAL_BENCHMARK_TOL:g}): {'holds' if len(hold) == 2 * len(VARIANTS) else 'FAILS'}")
print(f"the index's own one-way turnover: mean {index_turnover.mean():.4f} a month ({12 * index_turnover.mean():.1%} a year), median {index_turnover.median():.4f}, "
      f"largest {index_turnover.max():.3f} in {index_turnover.idxmax()}; months above the cap of {C.TURNOVER_CAP_MONTHLY_ONE_WAY}: {int((index_turnover > C.TURNOVER_CAP_MONTHLY_ONE_WAY).sum())}")

## Expectations 2 to 7: the measures

One row per path. Tracking errors and active returns in basis points a year; turnover as a fraction of the portfolio per month; the bias statistic under the 21 rule and under the ten-year variance ratio (daily variants only).

In [ ]:
rows = []
for (v, cs), path in paths.items():
    s = E.summarise(path, vr_scale=vr10 if v[1] == "daily_3y" or v == C.ROBUST_VARIANT else None)
    rows.append({"variant": LABEL[v], "set": cs, **s})
table = pd.DataFrame(rows).set_index(["variant", "set"])
bp_cols = ["realised TE", "forecast TE (mean)", "active return (gross)"] + [c for c in table.columns if c.startswith(("cost at", "net active return"))]
show = table.copy()
show[bp_cols] = 1e4 * show[bp_cols]
show = show.rename(columns={c: f"{c} (bp)" for c in bp_cols})
order = ["C0", "C1", "C2", "C3", "C3 tilt only"]
show = show.reindex(pd.MultiIndex.from_product([[LABEL[v] for v in VARIANTS], order], names=["variant", "set"]))
print(show[["realised TE (bp)", "forecast TE (mean) (bp)", "bias", "bias (variance ratio)", "turnover (one-way)", "relaxed months", "inaccurate months", "effective number", "industries held"]].to_string(float_format=lambda x: f"{x:.3f}"))
print()
print(show[["active return (gross) (bp)", "cost at 50 bp (bp)", "net active return at 50 bp (bp)", "information ratio at 50 bp", "net active return at 100 bp (bp)", "information ratio at 100 bp"]].to_string(float_format=lambda x: f"{x:.2f}"))

In [ ]:
daily = [LABEL[v] for v in CV.EVALUATION_VARIANTS if v[1] == "daily_3y"]
monthly = LABEL[("sample", "monthly_120")]
robust_label = LABEL[C.ROBUST_VARIANT]
te3 = table.xs("C3", level="set")["realised TE"]
rng = te3[daily].max() - te3[daily].min()
e2a = rng < C.EVAL_TE_RANGE_MAX
e2b = te3[monthly] > te3[daily].min()
print(f"expectation 2: realised tracking error under C3, daily estimators {1e4 * te3[daily].min():.1f} to {1e4 * te3[daily].max():.1f} bp (range {1e4 * rng:.1f}, level {1e4 * C.EVAL_TE_RANGE_MAX:.0f}): {'MET' if e2a else 'NOT MET'}; "
      f"monthly sample covariance {1e4 * te3[monthly]:.1f} bp, above the lowest daily: {'yes' if e2b else 'no'}")
bias3 = table.xs("C3", level="set")[["bias", "bias (variance ratio)"]]
lo, hi = C.EVAL_BIAS_RANGE
e3 = int(((bias3.loc[daily, "bias"] >= lo) & (bias3.loc[daily, "bias"] <= hi)).sum())
closer = int((abs(bias3.loc[daily, "bias (variance ratio)"] - 1) < abs(bias3.loc[daily, "bias"] - 1)).sum())
print(f"expectation 3: bias statistic under C3 and the 21 rule inside {lo} to {hi} for {e3} of {len(daily)} daily estimators ({', '.join(f'{b:.2f}' for b in bias3.loc[daily, 'bias'])}): {'MET' if e3 == len(daily) else 'NOT MET'}; "
      f"under the ten-year variance ratio ({', '.join(f'{b:.2f}' for b in bias3.loc[daily, 'bias (variance ratio)'])}), closer to 1 for {closer} of {len(daily)}")
singles = te3[daily + [monthly]]
e4 = te3[robust_label] <= singles.max() + 1e-12
print(f"expectation 4: the robust portfolio's realised tracking error under C3 is {1e4 * te3[robust_label]:.1f} bp against {1e4 * singles.min():.1f} (best single, {singles.idxmin()}) and {1e4 * singles.max():.1f} (worst, {singles.idxmax()}): {'MET' if e4 else 'NOT MET'}")
fix = pd.DataFrame({"mandate C3": te3, "tilt only": table.xs("C3 tilt only", level="set")["realised TE"]})
fix["difference"] = fix["mandate C3"] - fix["tilt only"]
e5 = int(((fix["difference"] > 0) & (fix["difference"] < C.OPT_FIX_RISE_MAX)).sum())
print(f"expectation 5: the mandate's C3 against the tilt alone, realised tracking error (bp): " + "; ".join(f"{k} {1e4 * r['mandate C3']:.1f} vs {1e4 * r['tilt only']:.1f} ({1e4 * r['difference']:+.1f})" for k, r in fix.iterrows())
      + f"; positive and below {1e4 * C.OPT_FIX_RISE_MAX:.0f} bp in {e5} of {len(fix)}: {'MET' if e5 == len(fix) else 'NOT MET'}")
relax = table.loc[(slice(None), ["C2", "C3"]), "relaxed share"]
e6 = bool((relax < C.EVAL_RELAX_SHARE_MAX).all())
print(f"expectation 6: share of months with a relaxed cap under C2 and C3, {relax.min():.3f} to {relax.max():.3f} (level {C.EVAL_RELAX_SHARE_MAX}): {'MET' if e6 else 'NOT MET'}")
gross = table["active return (gross)"]
years = len(months) / 12
tilt_gross, mandate_gross = gross.xs("C3 tilt only", level="set"), gross.xs("C3", level="set")
se_tilt = table.xs("C3 tilt only", level="set")["realised TE"] / np.sqrt(years)
mandate_cost = mandate_gross - tilt_gross
print(f"expectation 7: gross active return a year, the tilt alone {1e4 * tilt_gross.min():+.1f} to {1e4 * tilt_gross.max():+.1f} bp "
      f"(standard error of a mean over {years:.0f} years, the realised tracking error over the square root of {years:.0f}: {1e4 * se_tilt.min():.0f} to {1e4 * se_tilt.max():.0f}); "
      f"the mandate {1e4 * mandate_gross.min():+.1f} to {1e4 * mandate_gross.max():+.1f}; the mandate minus the tilt alone on the same months, variant by variant: "
      + ", ".join(f"{k} {1e4 * v:+.1f}" for k, v in mandate_cost.items()))
relaxed_months = pd.concat({cs: paths[(("sample", "daily_3y"), cs)]["relaxed"] for cs in ("C2", "C3")}, axis=1)
by_cal = relaxed_months.groupby(relaxed_months.index.month).sum()
by_cal.index = [pd.Timestamp(2000, m, 1).strftime("%b") for m in by_cal.index]
print("relaxed months by calendar month (sample covariance on daily data):")
print(by_cal.T.to_string())
first_relaxed = [cs for cs in ("C3", "C3 tilt only") if bool(paths[(("sample", "daily_3y"), cs)]["relaxed"].iloc[0])]
print(f"the first month is relaxed under: {', '.join(first_relaxed) if first_relaxed else 'none'}")

## By decade: realised against forecast tracking error under the mandate

For each variant under C3, the realised tracking error of each decade beside the mean forecast of the same months, and the ratio of the two; the ten-year variance ratio's version of the forecast beside it for the daily variants.

In [ ]:
first_label, last_label = f"{months[0]} to 1989-12", f"2020-01 to {months[-1]}"
blocks = []
for v in VARIANTS:
    path = paths[(v, "C3")]
    dec = E.by_decade(path, first_label, last_label).rename("realised TE")
    grp = pd.Series([first_label if m.year < 1990 else (last_label if m.year >= 2020 else f"{(m.year // 10) * 10}s") for m in path.index], index=path.index)
    fc = path.groupby(grp)["forecast_te"].mean().rename("forecast TE")
    fc_vr = (path["forecast_te"] * np.sqrt(vr10.loc[path.index])).groupby(grp).mean().rename("forecast TE (variance ratio)") if (v[1] == "daily_3y" or v == C.ROBUST_VARIANT) else fc.rename("forecast TE (variance ratio)") * np.nan
    d = pd.concat([dec, fc, fc_vr], axis=1)
    d["realised / forecast"] = d["realised TE"] / d["forecast TE"]
    d.insert(0, "variant", LABEL[v])
    blocks.append(d)
decades = pd.concat(blocks).set_index("variant", append=True).swaplevel()
show_d = decades.copy()
for c in ["realised TE", "forecast TE", "forecast TE (variance ratio)"]:
    show_d[c] = 1e4 * show_d[c]
print("under C3, by decade (bp a year):")
print(show_d.to_string(float_format=lambda x: f"{x:.1f}"))

## Expectations 8 and 9: RiskMetrics, and the forecasts calibrated on their own past

Two additions after the first run (6 October 2026), each with its expectation in `constants.py` before the code. First, RiskMetrics as a seventh variant: Dom, Howard, Jansen and Lohre (2024) find that weighting recent days more does more for a covariance than shrinkage or factor structure and that this simple dynamic estimator does as well as the DCC model for minimum-variance portfolios; notebook 10 found it gave the lowest unconstrained volatility and the best-calibrated forecast; the first design of this notebook left it out, a design miss. Second, a calibration of the forecasts, the fix a risk-model vendor applies when the bias statistic drifts from 1: the forecast of month t is multiplied by the ratio of realised to forecast tracking error over the previous 60 months of the same path, computed from months before t only, so nothing from the future enters it. One setting, 60 months, long enough for a stable ratio (a standard deviation estimated from 60 observations has a standard error of about 9%) and short enough to follow a change of regime within five years. The calibration cannot repair a month that no past resembles; it can repair a bias that persists.

In [ ]:
# Added after the first run (6 October 2026): expectation 8, RiskMetrics against the sample covariance on daily data.
ewma_label, sample_label = LABEL[C.EWMA_VARIANT], LABEL[("sample", "daily_3y")]
e8 = te3[ewma_label] <= te3[sample_label] + 1e-12
print(f"expectation 8: realised tracking error under C3, RiskMetrics {1e4 * te3[ewma_label]:.1f} bp against the sample covariance on daily data {1e4 * te3[sample_label]:.1f} bp: {'MET' if e8 else 'NOT MET'}; "
      f"RiskMetrics forecast {1e4 * table.loc[(ewma_label, 'C3'), 'forecast TE (mean)']:.1f} bp, bias {table.loc[(ewma_label, 'C3'), 'bias']:.2f}, turnover {table.loc[(ewma_label, 'C3'), 'turnover (one-way)']:.3f} a month, "
      f"net active return at 50 bp {1e4 * table.loc[(ewma_label, 'C3'), 'net active return at 50 bp']:.1f} bp")

In [ ]:
# Added after the first run (6 October 2026): expectation 9, the forecasts calibrated on their own past.
first_label, last_label = f"{months[0]} to 1989-12", f"2020-01 to {months[-1]}"
rows = []
for v in VARIANTS:
    path = paths[(v, "C3")]
    cal = E.calibrated_forecast(path)
    active = path["active_return"].to_numpy()
    grp = pd.Series([first_label if m.year < 1990 else (last_label if m.year >= 2020 else f"{(m.year // 10) * 10}s") for m in path.index], index=path.index)
    row = {"variant": LABEL[v], "bias, 21 rule": E.bias_statistic(active, path["forecast_te"].to_numpy()), "bias, calibrated": E.bias_statistic(active, cal.to_numpy()),
           "mean calibration factor": float((cal / path["forecast_te"]).mean())}
    for g in grp.unique():
        m = (grp == g).to_numpy()
        row[f"calibrated bias, {g}"] = E.bias_statistic(active[m], cal.to_numpy()[m])
    rows.append(row)
calib = pd.DataFrame(rows).set_index("variant")
print("the bias statistic under C3 before and after calibration on the path's own past (60-month window):")
print(calib.to_string(float_format=lambda x: f"{x:.2f}"))
lo9, hi9 = C.EVAL_CALIBRATED_BIAS_RANGE
e9 = int(((calib.loc[daily, "bias, calibrated"] >= lo9) & (calib.loc[daily, "bias, calibrated"] <= hi9)).sum())
print()
print(f"expectation 9: the calibrated bias statistic lies in {lo9} to {hi9} for {e9} of {len(daily)} daily estimators ({', '.join(f'{b:.2f}' for b in calib.loc[daily, 'bias, calibrated'])}): {'MET' if e9 == len(daily) else 'NOT MET'}")

## Save

The measures table, the by-decade table and the active returns of every path are written to `outputs/`, with the vintage in the file name; notebook 13 rebuilds the paths it attributes from the modules.

In [ ]:
os.makedirs(C.OUTPUT_DIR, exist_ok=True)
table.to_csv(f"{C.OUTPUT_DIR}/rolling_evaluation_{vintage}.csv", float_format="%.8f")
decades.to_csv(f"{C.OUTPUT_DIR}/rolling_evaluation_by_decade_{vintage}.csv", float_format="%.8f")
calib.to_csv(f"{C.OUTPUT_DIR}/rolling_evaluation_calibration_{vintage}.csv", float_format="%.8f")
active = pd.DataFrame({f"{LABEL[v]} | {cs}": p["active_return"] for (v, cs), p in paths.items()})
active.to_csv(f"{C.OUTPUT_DIR}/active_returns_{vintage}.csv", float_format="%.10f")
print("written:", sorted(f for f in os.listdir(C.OUTPUT_DIR) if f.startswith(("rolling", "active"))))

## What this notebook established, and what could be wrong

| Expectation | Result | Verdict |
|---|---|---|
| 1. Count first | 566 months, 35 paths with 566 records each; the benchmark held in every month under C0 and C1 in 14 of 14 paths (largest active return 2e-7 a month); the index's own turnover 0.39% a month | holds |
| 2. Does the estimator matter under the mandate | realised tracking error 84.7 to 93.4 basis points across the four daily estimators (range 8.8, level 10); the monthly sample covariance 95.8, above the lowest | MET |
| 3. Calibration of the forecasts under the mandate | bias statistics 1.22 to 1.39 under the 21 rule, 1 of 4 inside 0.8 to 1.25; 1.20 to 1.36 under the ten-year variance ratio, closer to 1 for 4 of 4 | NOT MET |
| 4. The robust portfolio | 87.4 basis points, against 84.7 for the best single estimator and 95.8 for the worst; its forecast the best calibrated (bias 1.09) | MET |
| 5. The realised cost of the fixes | the mandate's C3 above the tilt alone by 4.4 to 5.2 basis points for all seven variants | MET |
| 6. The cap | relaxed in 1 to 4 of 566 months (0.2% to 0.7%), the first month and Julys | MET |
| 7. Active return, costs, holdings | the tilt had no opportunity cost: +1.8 to +6.9 basis points a year gross, inside one standard error (12 to 13) of zero; the mandate -1.0 to -6.9; the mandate's two additions cost 8 to 10 against the tilt alone on the same months; turnover costs 5 to 9 at 50 basis points; net -6 to -15; information ratios -0.07 to -0.17 | reported |
| 8. RiskMetrics (added 6 October) | 84.3 basis points against 84.7 for the sample covariance on daily data; a difference of 0.4, inside the noise | MET, direction only |
| 9. Calibrated forecasts (added 6 October) | calibrated bias statistics 1.11, 1.11, 1.17 and 1.15 for the four daily estimators, 3 of 4 inside 0.85 to 1.15; the monthly covariance's 1.61 falls to 1.12; by decade 0.95 to 1.18 after 1989, 1.24 to 1.50 before | NOT MET |

Expectation 1 holds. 566 evaluation months, 35 paths, 566 records each; under C0 and C1 every path is the benchmark to the solver's precision (the largest active return in any month is 2e-7, two hundred-thousandths of a basis point; the level of 1e-10 written first was set for exact arithmetic and was raised to 1e-6 before the recorded run, as `constants.py` says). The index's own one-way turnover, the trading an index fund cannot avoid, averages 0.39% a month, 4.7% a year, with a median of 0.16% and a largest value of 13.3% in 1988-07; it exceeds 2% in 24 months, all of them Julys, when French rebuilds the industry portfolios from the end-of-June SIC codes. Under C2 the active weight bound leaves room to absorb those months, so the cap was relaxed once and the realised tracking error of following the index at 2% a month is 6.3 to 6.5 basis points.

Expectation 2, the extension's question, is met: under the mandate the four daily estimators realise 84.7 (sample), 84.8 (shrinkage), 91.5 (component model) and 93.4 (factor model) basis points a year of tracking error, a range of 8.8 against the level of 10, and the sample covariance of 120 monthly returns realises 95.8, above all four. The ranking is notebook 10's: on three years of daily data the sample covariance and shrinkage are the same estimate (shrinkage intensity 0.012) and the best, the two structured models pay 7 to 9 basis points for assuming that industries share nothing beyond their factors, and the monthly covariance pays for its noise. The pattern of Dom, Howard, Jansen and Lohre (2024) holds in tracking-error terms with a qualification: the constraints shrink the differences between estimators to about a tenth of the tracking error, and do not remove them.

Expectation 3 is not met, and the way it fails is the notebook's main finding about forecasts. Over the 566 months the daily estimators' forecasts under-forecast the realised tracking error by 22% to 39% (bias statistics 1.22 to 1.39) and the monthly covariance's by 61%. The ten-year variance ratio, which repaired the benchmark's forecasts in notebook 10, changes these by 0.02 to 0.03, because its mean over the evaluation months is 1.00: the positive autocorrelation of the early decades and the negative autocorrelation of the later ones cancel over this sample, and the active portfolio's own autocorrelation is not the benchmark's. The by-decade table says where the miss is. In 1979-07 to 1989-12 the realised tracking error was 1.6 to 2.0 times the forecast for every variant (138 to 156 basis points against 76 to 90); in the 1990s 1.3 to 1.4 times; in the 2000s 1.1 to 1.2; in the 2010s and from 2020 the forecasts were right to within 10%. The first decade is the oil shock. Oil was 15% of the benchmark and the tilt removed 7.6 percentage points of it; the oil price doubled in 1979 and 1980 and collapsed in 1986, and in single months the portfolio's active return reached +1.45% (1980-07), +1.41% (1980-02) and -1.14% (1981-03) against a forecast monthly standard deviation of about 0.24%, five to six forecast standard deviations. A three-year window of daily returns estimated on 1976 to 1979 cannot know that Oil's own volatility is about to triple; this is a change of regime, which no window length repairs, and it is why the realised number is the one that counts. The second part of the miss is of a different kind and does not go away with the decades: the monthly sample covariance under-forecasts by 30% to 70% in every decade, the daily estimators by 10% to 40% in the first three. An optimiser handed a covariance with errors finds the portfolio that looks least risky in that covariance, and part of what makes it look least risky is the errors; the forecast is then an in-sample minimum and the realised number is out of sample. This is the optimisation bias of Michaud (1989), the error-maximisation property of optimised portfolios, and it is larger the noisier the covariance, which is why the monthly covariance's forecast is the worst calibrated (1.61) and the robust portfolio's worst-case forecast, the maximum over five matrices, the best (1.09): taking the worst case across estimates offsets the optimism of each. For a manager: a tracking-error forecast that comes out of an optimiser is biased low, here by a fifth to a third with a good covariance, and the worst case across estimators is the forecast whose bias statistic is nearest one.

Whether the miss of expectation 3 was foreseeable. Its direction was, and the design did not use what it knew. Notebook 10's by-decade table had the benchmark's own forecasts under-forecasting by 34% in 1979 to 1989 under the 21 rule, and Michaud (1989) says that an optimised portfolio's forecast risk is biased low. Both were known on 4 October when the range 0.8 to 1.25 was written for the whole sample, as if the active portfolio's forecasts would be as well calibrated on average as the benchmark's. The miss is on the design side: the expectation should have been written by decade, with a statistic above 1 expected for the first decade and for every optimised portfolio. The size of the miss was not foreseeable from the literature: a first decade at 1.6 to 2.0 times the forecast is the oil shock acting on a portfolio tilted away from Oil, and notebook 10's table, which showed 1.34 for the benchmark, had no active portfolio in it. Expectation 9 below adds the fix for the part of the bias that persists; it uses only months before each forecast, so it is not fitted on the months it is judged on.

Expectation 4 holds. The robust portfolio realises 87.4 basis points, 2.7 above the best single estimator and 8.4 below the worst, at a one-way turnover of 1.1% a month against 1.3% for the sample covariance. It is nobody's worst case, which is what it was built for, and its forecast is the one a manager could have trusted.

Expectation 5 holds. The mandate's fixes, beta neutrality and the exclusion of tobacco and weapons, cost 4.4 to 5.2 basis points a year of realised tracking error for every one of the seven variants. Notebook 11 had forecast 2 to 7 for the daily estimators and 19 to 25 for the monthly covariance; the realised cost for the monthly covariance is 4.8, so the large forecast was the noise talking, the same noise that made its hedges attractive in the first place.

Expectation 6 holds. The cap was relaxed in 2 of 566 months under C3 for the daily estimators and RiskMetrics (the first month, when the tilt is put on from the index, and one July), 3 for the robust portfolio, 4 for the monthly covariance, and once under C2; the share is 0.2% to 0.7% against a level of 10%. The months are the ones the design named.

Expectation 7, the active returns. Opportunity cost, the return a fund gives up by obeying a constraint, is the first thing a client asks about a tilt, and the tilt portfolio's returns show that there was none: over 47 years the tilt alone earned between +1.8 and +6.9 basis points a year gross against the benchmark, and a mean active return over 47 years has a standard error of the realised tracking error divided by the square root of 47, 80 to 91 basis points divided by 6.9, 12 to 13 basis points a year, so the figure is inside one standard error of zero. Holding Coal, Oil and Utilities at half their index weight gave up no return that can be told from chance, which for a sustainability mandate is the result its holders want. The mandate earned between -1.0 and -6.9, also inside one standard error of zero. The difference between the mandate and the tilt alone, 8 to 10 basis points a year, is the opportunity cost of the mandate's two additions, beta neutrality and holding tobacco and weapons at zero, two industries that outperformed the market over the sample; it is a difference of two means on the same months and is less noisy than either, and it is the number a manager reports to the client who asked for the exclusions; notebook 13 splits it between the factors and the industries' own returns and gives its standard error. Trading costs at 50 basis points per unit of turnover take 5 to 9 a year; they are arithmetic on the turnover and carry no noise. Net of them the mandate's portfolios trail the benchmark by 6 to 15 basis points a year, with information ratios of -0.07 to -0.17, on a tracking error of 85 to 96. A manager of such a mandate is judged on four numbers before the active return: the realised tracking error against the ex-ante limit (85 to 96 basis points against 100: inside, for every variant), the constraints obeyed in every month (the relaxed-cap months counted and explained), the turnover and its cost (0.9% to 1.5% a month, 5 to 9 basis points a year at 50), and the cost of each exclusion reported to the client. The information ratio measures skill at forecasting returns; this portfolio makes no return forecast, so an information ratio inside one standard error of zero is the outcome the design implies. The effective number of holdings is 19.0 to 19.8 against the benchmark's 18.9, and the portfolios hold 41 to 45 of the 49 industries. Where the active return came from, by factor, is notebook 13's question.

Expectation 8, RiskMetrics, is met in direction and says nothing in size. Under the mandate RiskMetrics realises 84.3 basis points against 84.7 for the sample covariance on three years of daily data, a difference of 0.4 basis points on paths that share most of their months; a realised tracking error of 85 basis points measured on 566 months has a standard error of about 2.5, so the two are the same number. Its forecast is as low as the others' (bias 1.23 against 1.26), and its turnover is the highest of the daily variants (1.4% a month against 0.9% to 1.3%), because a covariance that weights recent days most changes more from month to month and the portfolio follows it; after costs at 50 basis points it nets -13.9 against -14.8. Dom, Howard, Jansen and Lohre (2024) found that time dynamics were the design choice that mattered most for a constrained minimum-variance portfolio of single stocks; notebook 10 found the same for the unconstrained minimum-variance portfolio of the 49 industries. Under the tracking-error mandate on 49 industries the gain is gone: the active weight bound of 2 percentage points and the turnover cap leave the optimiser too little room for a better covariance to show. The expectation was written as "at or below", and a result of equality meets it without teaching anything; written again, it would name a smallest difference worth calling a gain, 2.5 basis points, one standard error.

Expectation 9, the calibrated forecasts, is not met by one estimator and 0.02. Multiplying each forecast by the ratio of realised to forecast tracking error over the previous 60 months of the same path moves every variant's bias statistic towards 1: the daily estimators from 1.22 to 1.39 to 1.11 to 1.17, the monthly covariance from 1.61 to 1.12, RiskMetrics from 1.23 to 1.08, the robust portfolio from 1.09 to 1.06. Three of the four daily estimators land inside 0.85 to 1.15 (1.11, 1.11, 1.15); the factor model lands at 1.17. The by-decade columns say what the calibration repairs and what it cannot. From 1990 on, the calibrated statistics of the daily estimators lie between 0.95 and 1.18 in every decade, and the monthly covariance's between 1.01 and 1.18: the persistent part of the bias, the optimisation bias, is a ratio that the previous five years measure well. In 1979 to 1989 the calibrated statistics are still 1.24 to 1.50, because the first 24 months run uncorrected and the shock enters the realised numbers before it enters the window; a correction built from the past arrives after the regime it is correcting for. The mean calibration factor, 1.16 to 1.30 for the daily estimators and 1.52 for the monthly covariance, is the size of the correction a manager would have applied in real time. The range 0.85 to 1.15 was written for the whole sample, which includes the 126 months no backward-looking calibration can repair; written for the months from 1990, where the calibration has a full window, the four daily estimators would have landed between 0.96 and 1.18, and the verdict on that range is not claimed, because the range was not written first.

The robust portfolio's status column reports 154 and 113 months of inaccurate solver status under C0 and C1 and one under C3. Under C0 and C1 the optimum is the benchmark exactly, where the worst-case tracking error is zero and the problem's cone constraints are all at their tips; an interior-point method reaches that point only approximately and says so. The weights it returned passed every feasibility check, and the active return in those months is below 2e-7.

**What this notebook does not settle.**

- The under-forecast of 1979 to 1989 is a change of regime that no estimation window anticipates and that the calibration of expectation 9 reaches only after the fact. A manager handles it with scenario and stress tests outside the covariance, which the extension does not build; the notebook states the size of the miss so that a reader knows what a covariance-based forecast can and cannot do.
- The calibration of expectation 9 uses one window length, 60 months, fixed before its code. A shorter window would follow a change of regime sooner and carry more noise (a ratio from 24 months has a standard error of about 15%); the trade-off is stated and not searched, because searching it on these 566 months would fit the window to the sample.
- Where the tilt's and the exclusions' active returns came from, by Fama-French factor, is notebook 13's question.
- The result holds for 49 industries, whose benchmark is concentrated (effective number 19); at the level of single stocks the differences between estimators, and the gain from time dynamics that expectation 8 did not find here, may be larger, which is the second study's question.
- Trading costs are one proportional rate for every industry and month; the realised numbers are gross of costs, and the net figures use 50 and 100 basis points per unit of one-way turnover as the replication does.
- The level for expectation 1 was raised from 1e-10 to 1e-6 after the first run, for the solver's precision. The 30 paths of the design without RiskMetrics and the calibration give the same numbers to every printed decimal with them in the notebook as without.